# 04c Extraction, batch c: twelve included studies coded from their PDFs

Design log.
- Batch c of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a and b were. The twelve reports: Contextual Integrity Verification (2508.09288), a 2022 study of adversarial fine-tuning (2509.14271), SecInfer (2509.24967), RL-Hammer (2510.04885), ARLAS (2510.05442), The Attacker Moves Second (2510.09023), VerIH (2511.04694, read as the arXiv version; the search record lists Findings of ACL 2026), Prompt Fencing (2511.19727), Label Disguise Defense (2511.21752), surface heuristics in trained defences (2601.07185; the search record lists ACL 2026), ReasAlign (2601.10173) and RedVisor (2602.01795).
- The next report in list order, 2601.04666, is held back: its PDF (24.6 MB) is larger than the Drive connector used to read the PDFs can fetch. RedVisor took its place, and 2601.04666 will be coded in a later batch once the PDF can be read.
- Each record's extraction_scope names the tables and rows coded. As in batch b, detectors, sanitisers and rewriters that act before an unchanged model (PromptArmor, PromptLocate and DataFilter in RedVisor; the guardrail comparators in Contextual Integrity Verification) are left out and the scope says so (amendment 4).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - SecInfer samples several responses and picks one; that sampling is part of the defence, so the decoding condition of its comparisons is matched_unstated rather than unmatched. ReasAlign's judge-guided beam search and RedVisor's two-phase generation are treated the same way.
  - VerIH trains with reinforcement learning from verifiable rewards. Axis 2 has no value for non-adversarial reinforcement learning, so it is coded preference_optimisation and a residual note says so.
  - ReasAlign's training targets were written by GPT-4o-mini, so axis 2 includes distillation, as Jatmo's does in batch a. Its preference optimisation trains a separate judge used in the test-time search, not the answering model; a residual note says so.
  - ReasAlign's abstract gives Meta SecAlign 74.4% attack success on CyberSecEval2. The paper's own Figure 4 prints 7.3 for that bar; 74.4 is the undefended model on SEP. The records follow the figure and a residual note records the mismatch.
  - RedVisor's training data uses the same five attack categories as its main test table, while the text says the tested patterns were unseen; the payload string is new at test time. A residual note records both statements with their pages.
  - Values printed as bar labels in ReasAlign's Figures 3 and 4 are coded figure_only with the printed value kept, as in batch a; Stage 3 checks each against the page text. Two labels (94.4 and 96.1) run into the next label in every text extraction, so they are coded without a value, carry no comparison and are listed in a residual note for a human check. RedVisor's utility chart prints no labels, so only the three win rates stated in its text carry values.
  - The Label Disguise Defense compares alias labels with native labels under the same few-shot demonstrations, so its comparisons are component (M) comparisons. The surface-heuristics study reports no attack success at all; it is coded for benign accuracy and refusal of benign tasks.
  - One comparison grades U: Contextual Integrity Verification's ablation does not say which attack prompts it used.
- Change to Stage 3: a chart label not found in the plain or the column-preserving text is now also looked for in the third extraction (pypdfium2), as printed table values already were in batch b. Labels found that way are listed in reports/extraction_04c_check.csv and counted separately in the decision record.
- Full-text exclusions found while coding this batch: none.
- Outputs: reports/records/ (twelve files), reports/extraction_04c_check.csv, reports/extraction_04c_summary.csv, reports/extraction_batches.csv (rows for batch c added, earlier batches kept), reports/decision_04c.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 12 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`civ_arxiv2025` (2508.09288)

In [ ]:
%%writefile reports/records/civ_arxiv2025.yaml
report:
  id: civ_arxiv2025
  slug: '2508.09288'
  title: 'Can AI Keep a Secret? Contextual Integrity Verification: A Provable Security Architecture for LLMs'
  first_author: Aayush Gupta
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2508.09288'
  version_read: arXiv v2
  first_date: '2025-08-12'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Sections 4 and 5: Table 2 rows None and CIV (Elite ASR, SoK ASR, Sim) and the Unsigned labels and Hard mask + gate rows of Table 3; the abstract claim (the guardrail comparators, which sit outside the
    model, latency and the formal analysis not coded)'
defences:
- id: civ_l3
  name: Contextual Integrity Verification (CIV)
  origin: this_report
  base_checkpoint: Llama-3-8B Instruct (frozen)
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: measured
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
  components:
  - every token carries an HMAC-signed trust label (SYSTEM > USER > TOOL > DOC > WEB)
  - attention from higher-trust to lower-trust tokens masked to minus infinity in 20 replaced decoder layers
  - optional gate that damps feed-forward updates of tokens with inaccessible higher-trust context
  layout: token provenance labels assigned by the serving stack; no prompt change
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: not applicable (frozen model)
  anchor: We deploy Llama-3-8B Instruct as our baseline, surgically replacing
  signal_dependence_detail: 'Table 3: unsigned labels give 7.1% ASR and no trust propagation 3.2%, against 0% for the full system'
  pages:
    components: 7
    signal_dependence: 9
    isolation: 9
- id: civ_unsigned_l3
  name: CIV with unsigned labels (no HMAC), ablation
  origin: this_report
  base_checkpoint: Llama-3-8B Instruct (frozen)
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  anchor: Unsigned labels (no HMAC)
evaluations:
- id: ev_none_elite
  defence_id: none:Llama-3-8B Instruct (frozen)
  benchmark: 'Elite-Attack: 100 verified jailbreak and injection prompts in ten families (including indirect web injection)'
  attacker_type: static
  attacker_access: black_box
  judge: attack success rate (criterion not stated)
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 8
  baseline_kind: off_the_shelf
- id: ev_none_sok
  defence_id: none:Llama-3-8B Instruct (frozen)
  benchmark: 'SoK-246: 246 prompts curated by Hu et al.'
  attacker_type: static
  attacker_access: black_box
  judge: attack success rate (criterion not stated)
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  sample_size: '246'
  decoding: not reported
  pages:
    benchmark: 8
  baseline_kind: off_the_shelf
- id: ev_none_benign
  defence_id: none:Llama-3-8B Instruct (frozen)
  benchmark: 'Benign-10: ten legitimate task categories'
  attacker_type: none
  attacker_access: not_applicable
  judge: token-level exact-match similarity to the undefended model's output
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_civ_elite
  defence_id: civ_l3
  benchmark: 'Elite-Attack: 100 verified jailbreak and injection prompts in ten families (including indirect web injection)'
  attacker_type: static
  attacker_access: black_box
  judge: attack success rate (criterion not stated)
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 8
- id: ev_civ_sok
  defence_id: civ_l3
  benchmark: 'SoK-246: 246 prompts curated by Hu et al.'
  attacker_type: static
  attacker_access: black_box
  judge: attack success rate (criterion not stated)
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  sample_size: '246'
  decoding: not reported
  pages:
    benchmark: 8
- id: ev_civ_benign
  defence_id: civ_l3
  benchmark: 'Benign-10: ten legitimate task categories'
  attacker_type: none
  attacker_access: not_applicable
  judge: token-level exact-match similarity to the undefended model's output
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_abl_full
  defence_id: civ_l3
  benchmark: attack prompts (set not stated for the ablation)
  attacker_type: static
  attacker_access: black_box
  judge: attack success rate (criterion not stated)
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  decoding: not reported
- id: ev_abl_unsigned
  defence_id: civ_unsigned_l3
  benchmark: attack prompts (set not stated for the ablation)
  attacker_type: static
  attacker_access: black_box
  judge: attack success rate (criterion not stated)
  layout_same_as_defence: 'yes'
  measured_by: civ_arxiv2025
  page: 9
  decoding: not reported
results:
- id: r_none_elite
  evaluation_id: ev_none_elite
  quantity: R1
  coverage: direct
  metric_as_printed: Elite ASR
  value: '54'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_none_sok
  evaluation_id: ev_none_sok
  quantity: R1
  coverage: direct
  metric_as_printed: SoK ASR
  value: '48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_none_sim
  evaluation_id: ev_none_benign
  quantity: general_utility
  coverage: proxy
  metric_as_printed: Sim
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_civ_elite
  evaluation_id: ev_civ_elite
  quantity: R1
  coverage: direct
  metric_as_printed: Elite ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_civ_sok
  evaluation_id: ev_civ_sok
  quantity: R1
  coverage: direct
  metric_as_printed: SoK ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_civ_sim
  evaluation_id: ev_civ_benign
  quantity: general_utility
  coverage: proxy
  metric_as_printed: Sim
  value: '93.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_abl_full
  evaluation_id: ev_abl_full
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 3
- id: r_abl_unsigned
  evaluation_id: ev_abl_unsigned
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '7.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 3
comparisons:
- id: c_elite
  left: ev_civ_elite
  right: ev_none_elite
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_sok
  left: ev_civ_sok
  right: ev_none_sok
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_benign
  left: ev_civ_benign
  right: ev_none_benign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_abl_hmac
  left: ev_abl_full
  right: ev_abl_unsigned
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unreported
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: HMAC-signed labels against unsigned labels
  page: 9
claims:
- id: k1
  text: deterministic, cryptographically verifiable non-interference at inference time on frozen models
  location: abstract
  anchor: deterministic, cryptographically verifiable non-interference guarantees at inference-time
  jobs:
  - R1
  supported_by:
  - c_elite
  - c_sok
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2
    general_utility: Table 2 (Sim, token match with the undefended output)

`advft2022_arxiv2025` (2509.14271)

In [ ]:
%%writefile reports/records/advft2022_arxiv2025.yaml
report:
  id: advft2022_arxiv2025
  slug: '2509.14271'
  title: 'Early Approaches to Adversarial Fine-Tuning for Prompt Injection Defense: A 2022 Study of GPT-3 and Contemporary Models'
  first_author: Gustavo Sandoval
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2509.14271'
  version_read: arXiv v1
  first_date: '2025-09-15'
  read: F
  role: proposes
  code_release: https://github.com/GusSand/PromptInject
  extraction_scope: 'whole report (8 pages): Table 1 coded (before and after adversarial fine-tuning, Ada, Babbage, Curie); the abstract claim (Table 2, models tested without the defence, and Figure 5 not coded)'
defences:
- id: aft_ada
  name: Adversarial fine-tuning with <userInput> tags
  origin: this_report
  base_checkpoint: text-ada-001
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - user input wrapped in <userInput> tags
  - OpenAI fine-tuning on translation, grammar, sentiment and summarisation samples, some with injected instructions inside the tags and completions that follow the original instruction
  layout: PROMPT + <userInput> USER_INPUT </userInput>
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: no (OpenAI fine-tuned models)
  anchor: user input gets wrapped in the user input tags
  pages:
    layout: 4
- id: aft_babbage
  name: Adversarial fine-tuning with <userInput> tags
  origin: this_report
  base_checkpoint: text-babbage-001
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - user input wrapped in <userInput> tags
  - OpenAI fine-tuning on translation, grammar, sentiment and summarisation samples, some with injected instructions inside the tags and completions that follow the original instruction
  layout: PROMPT + <userInput> USER_INPUT </userInput>
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: no (OpenAI fine-tuned models)
  anchor: user input gets wrapped in the user input tags
  pages:
    layout: 4
- id: aft_curie
  name: Adversarial fine-tuning with <userInput> tags
  origin: this_report
  base_checkpoint: text-curie-001
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - user input wrapped in <userInput> tags
  - OpenAI fine-tuning on translation, grammar, sentiment and summarisation samples, some with injected instructions inside the tags and completions that follow the original instruction
  layout: PROMPT + <userInput> USER_INPUT </userInput>
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  released_weights: no (OpenAI fine-tuned models)
  anchor: user input gets wrapped in the user input tags
  pages:
    layout: 4
evaluations:
- id: ev_ada_before_gh
  defence_id: none:text-ada-001
  benchmark: 'PromptInject: 35 task prompts with goal hijacking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five goal hijacking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
  baseline_kind: off_the_shelf
- id: ev_ada_before_pl
  defence_id: none:text-ada-001
  benchmark: 'PromptInject: 35 task prompts with prompt leaking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five prompt leaking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
  baseline_kind: off_the_shelf
- id: ev_ada_after_gh
  defence_id: aft_ada
  benchmark: 'PromptInject: 35 task prompts with goal hijacking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five goal hijacking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
- id: ev_ada_after_pl
  defence_id: aft_ada
  benchmark: 'PromptInject: 35 task prompts with prompt leaking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five prompt leaking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
- id: ev_babbage_before_gh
  defence_id: none:text-babbage-001
  benchmark: 'PromptInject: 35 task prompts with goal hijacking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five goal hijacking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
  baseline_kind: off_the_shelf
- id: ev_babbage_before_pl
  defence_id: none:text-babbage-001
  benchmark: 'PromptInject: 35 task prompts with prompt leaking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five prompt leaking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
  baseline_kind: off_the_shelf
- id: ev_babbage_after_gh
  defence_id: aft_babbage
  benchmark: 'PromptInject: 35 task prompts with goal hijacking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five goal hijacking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
- id: ev_babbage_after_pl
  defence_id: aft_babbage
  benchmark: 'PromptInject: 35 task prompts with prompt leaking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five prompt leaking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
- id: ev_curie_before_gh
  defence_id: none:text-curie-001
  benchmark: 'PromptInject: 35 task prompts with goal hijacking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five goal hijacking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
  baseline_kind: off_the_shelf
- id: ev_curie_before_pl
  defence_id: none:text-curie-001
  benchmark: 'PromptInject: 35 task prompts with prompt leaking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five prompt leaking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
  baseline_kind: off_the_shelf
- id: ev_curie_after_gh
  defence_id: aft_curie
  benchmark: 'PromptInject: 35 task prompts with goal hijacking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five goal hijacking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
- id: ev_curie_after_pl
  defence_id: aft_curie
  benchmark: 'PromptInject: 35 task prompts with prompt leaking strings (1,260 attack variations)'
  attacker_type: static
  attacker_access: black_box
  judge: Levenshtein similarity of the output to the target string above a threshold
  layout_same_as_defence: 'yes'
  measured_by: advft2022_arxiv2025
  page: 5
  attacker_budget: five prompt leaking strings
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
results:
- id: r_ada_before_gh
  evaluation_id: ev_ada_before_gh
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Goal Hijacking
  value: '26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ada_before_pl
  evaluation_id: ev_ada_before_pl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Prompt Leaking
  value: '2.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ada_after_gh
  evaluation_id: ev_ada_after_gh
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Goal Hijacking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_ada_after_pl
  evaluation_id: ev_ada_after_pl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Prompt Leaking
  value: '2.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_babbage_before_gh
  evaluation_id: ev_babbage_before_gh
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Goal Hijacking
  value: '31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_babbage_before_pl
  evaluation_id: ev_babbage_before_pl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Prompt Leaking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_babbage_after_gh
  evaluation_id: ev_babbage_after_gh
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Goal Hijacking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_babbage_after_pl
  evaluation_id: ev_babbage_after_pl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Prompt Leaking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_curie_before_gh
  evaluation_id: ev_curie_before_gh
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Goal Hijacking
  value: '18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_curie_before_pl
  evaluation_id: ev_curie_before_pl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Prompt Leaking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_curie_after_gh
  evaluation_id: ev_curie_after_gh
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Goal Hijacking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_curie_after_pl
  evaluation_id: ev_curie_after_pl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack success rate, Prompt Leaking
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
comparisons:
- id: c_ada_gh
  left: ev_ada_after_gh
  right: ev_ada_before_gh
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_ada_pl
  left: ev_ada_after_pl
  right: ev_ada_before_pl
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_babbage_gh
  left: ev_babbage_after_gh
  right: ev_babbage_before_gh
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_babbage_pl
  left: ev_babbage_after_pl
  right: ev_babbage_before_pl
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_curie_gh
  left: ev_curie_after_gh
  right: ev_curie_before_gh
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_curie_pl
  left: ev_curie_after_pl
  right: ev_curie_before_pl
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
claims:
- id: k1
  text: attack success falls from 31% to near zero for the smaller GPT-3 models
  location: abstract
  anchor: attack success rates were reduced to near zero for smaller GPT-3 variants
  jobs:
  - R1
  supported_by:
  - c_ada_gh
  - c_babbage_gh
  - c_curie_gh
  page: 1
  contradicted_by:
  - r_ada_after_pl
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1

`secinfer_arxiv2025` (2509.24967)

In [ ]:
%%writefile reports/records/secinfer_arxiv2025.yaml
report:
  id: secinfer_arxiv2025
  slug: '2509.24967'
  title: 'SecInfer: Preventing Prompt Injection via Inference-time Scaling'
  first_author: Yupei Liu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2509.24967'
  version_read: arXiv (version not printed on the PDF)
  first_date: '2025-09-29'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Sections 4 and 5.1 to 5.3: Table 2 rows No, Delimiters, Sandwich, Instructional, MetaSecAlign and SecInfer under the Combined Attack and rows MetaSecAlign and SecInfer under GCG, all six target tasks;
    Tables 5 and 6 (agents); the abstract claim (paraphrasing, retokenization, DS+PL and PromptArmor, which act outside the model, Table 1, Table 3 and the adaptive attacks of Table 8 not coded)'
defences:
- id: secinfer_l31
  name: SecInfer
  origin: this_report
  base_checkpoint: LLaMA3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  components:
  - N = 5 candidate responses, each sampled with a chain-of-thought system prompt drawn from a set, temperature and top-k sampling
  - 'closed-domain tasks: majority over responses mapped to the output space; open-domain: clustering of response embeddings and a judge LLM (Qwen3-8B) that picks the cluster aligned with the target task'
  layout: system prompt, then the target instruction, then the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: '4.2 Step I: System-prompt-guided Sampling'
  pages:
    components: 13
- id: delim_l31
  name: Delimiters
  origin: other_report
  base_checkpoint: LLaMA3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 14
  anchor: SecInfer is more effective than prevention-based defense baselines
  layout: system prompt, then the target instruction, then the data
- id: sandwich_l31
  name: Sandwich
  origin: other_report
  base_checkpoint: LLaMA3.1-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 14
  anchor: SecInfer is more effective than prevention-based defense baselines
  layout: system prompt, then the target instruction, then the data
- id: instr_l31
  name: Instructional
  origin: other_report
  base_checkpoint: LLaMA3.1-8B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 14
  anchor: SecInfer is more effective than prevention-based defense baselines
  layout: system prompt, then the target instruction, then the data
- id: msa_l31
  name: MetaSecAlign
  origin: other_report
  base_checkpoint: LLaMA3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 14
  anchor: SecInfer is more effective than prevention-based defense baselines
  layout: system prompt, then the target instruction, then the data
evaluations:
- id: ev_none_agnews_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: AG News target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_none_agnews_ca
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_none_mmlu_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: MMLU target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_none_mmlu_ca
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_none_gpqa_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: GPQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_none_gpqa_ca
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_none_narrqa_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: NarrativeQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_none_narrqa_ca
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_none_squad_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: SQuAD_v2 target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_none_squad_ca
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_none_simpleqa_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: SimpleQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
  baseline_kind: off_the_shelf
- id: ev_none_simpleqa_ca
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
  baseline_kind: off_the_shelf
- id: ev_delim_agnews_clean
  defence_id: delim_l31
  benchmark: AG News target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_delim_agnews_ca
  defence_id: delim_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_delim_mmlu_clean
  defence_id: delim_l31
  benchmark: MMLU target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_delim_mmlu_ca
  defence_id: delim_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_delim_gpqa_clean
  defence_id: delim_l31
  benchmark: GPQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_delim_gpqa_ca
  defence_id: delim_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_delim_narrqa_clean
  defence_id: delim_l31
  benchmark: NarrativeQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_delim_narrqa_ca
  defence_id: delim_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_delim_squad_clean
  defence_id: delim_l31
  benchmark: SQuAD_v2 target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_delim_squad_ca
  defence_id: delim_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_delim_simpleqa_clean
  defence_id: delim_l31
  benchmark: SimpleQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_delim_simpleqa_ca
  defence_id: delim_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_sandwich_agnews_clean
  defence_id: sandwich_l31
  benchmark: AG News target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_sandwich_agnews_ca
  defence_id: sandwich_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_sandwich_mmlu_clean
  defence_id: sandwich_l31
  benchmark: MMLU target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_sandwich_mmlu_ca
  defence_id: sandwich_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_sandwich_gpqa_clean
  defence_id: sandwich_l31
  benchmark: GPQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_sandwich_gpqa_ca
  defence_id: sandwich_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_sandwich_narrqa_clean
  defence_id: sandwich_l31
  benchmark: NarrativeQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_sandwich_narrqa_ca
  defence_id: sandwich_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_sandwich_squad_clean
  defence_id: sandwich_l31
  benchmark: SQuAD_v2 target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_sandwich_squad_ca
  defence_id: sandwich_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_sandwich_simpleqa_clean
  defence_id: sandwich_l31
  benchmark: SimpleQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_sandwich_simpleqa_ca
  defence_id: sandwich_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_instr_agnews_clean
  defence_id: instr_l31
  benchmark: AG News target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_instr_agnews_ca
  defence_id: instr_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_instr_mmlu_clean
  defence_id: instr_l31
  benchmark: MMLU target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_instr_mmlu_ca
  defence_id: instr_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_instr_gpqa_clean
  defence_id: instr_l31
  benchmark: GPQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_instr_gpqa_ca
  defence_id: instr_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_instr_narrqa_clean
  defence_id: instr_l31
  benchmark: NarrativeQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_instr_narrqa_ca
  defence_id: instr_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_instr_squad_clean
  defence_id: instr_l31
  benchmark: SQuAD_v2 target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_instr_squad_ca
  defence_id: instr_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_instr_simpleqa_clean
  defence_id: instr_l31
  benchmark: SimpleQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_instr_simpleqa_ca
  defence_id: instr_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_agnews_clean
  defence_id: msa_l31
  benchmark: AG News target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_msa_agnews_ca
  defence_id: msa_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_mmlu_clean
  defence_id: msa_l31
  benchmark: MMLU target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_msa_mmlu_ca
  defence_id: msa_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_gpqa_clean
  defence_id: msa_l31
  benchmark: GPQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_msa_gpqa_ca
  defence_id: msa_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_narrqa_clean
  defence_id: msa_l31
  benchmark: NarrativeQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_msa_narrqa_ca
  defence_id: msa_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_squad_clean
  defence_id: msa_l31
  benchmark: SQuAD_v2 target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_msa_squad_ca
  defence_id: msa_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_simpleqa_clean
  defence_id: msa_l31
  benchmark: SimpleQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_msa_simpleqa_ca
  defence_id: msa_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_secinfer_agnews_clean
  defence_id: secinfer_l31
  benchmark: AG News target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_secinfer_agnews_ca
  defence_id: secinfer_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_secinfer_mmlu_clean
  defence_id: secinfer_l31
  benchmark: MMLU target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_secinfer_mmlu_ca
  defence_id: secinfer_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_secinfer_gpqa_clean
  defence_id: secinfer_l31
  benchmark: GPQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_secinfer_gpqa_ca
  defence_id: secinfer_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_secinfer_narrqa_clean
  defence_id: secinfer_l31
  benchmark: NarrativeQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_secinfer_narrqa_ca
  defence_id: secinfer_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_secinfer_squad_clean
  defence_id: secinfer_l31
  benchmark: SQuAD_v2 target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_secinfer_squad_ca
  defence_id: secinfer_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_secinfer_simpleqa_clean
  defence_id: secinfer_l31
  benchmark: SimpleQA target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: target-task metric without attack (per-task metrics, Appendix F)
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
- id: ev_secinfer_simpleqa_ca
  defence_id: secinfer_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: static
  attacker_access: black_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: Combined Attack
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    benchmark: 13
- id: ev_msa_agnews_gcg
  defence_id: msa_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_msa_mmlu_gcg
  defence_id: msa_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_msa_gpqa_gcg
  defence_id: msa_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_msa_narrqa_gcg
  defence_id: msa_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_msa_squad_gcg
  defence_id: msa_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_msa_simpleqa_gcg
  defence_id: msa_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_secinfer_agnews_gcg
  defence_id: secinfer_l31
  benchmark: AG News target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_secinfer_mmlu_gcg
  defence_id: secinfer_l31
  benchmark: MMLU target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_secinfer_gpqa_gcg
  defence_id: secinfer_l31
  benchmark: GPQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_secinfer_narrqa_gcg
  defence_id: secinfer_l31
  benchmark: NarrativeQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_secinfer_squad_gcg
  defence_id: secinfer_l31
  benchmark: SQuAD_v2 target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_secinfer_simpleqa_gcg
  defence_id: secinfer_l31
  benchmark: SimpleQA target task, 8 injected tasks, 100 contaminated samples per pair
  attacker_type: optimisation
  attacker_access: white_box
  judge: 'ASR: response completes the injected task; UA: target-task metric under attack (per-task metrics, Appendix F)'
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 15
  attacker_budget: GCG-based attack on the target instruction and backend LLM
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  pages:
    judge: 13
    attacker_access: 12
- id: ev_none_ia_dh_base
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: InjecAgent, Direct Harm, Base Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  baseline_kind: off_the_shelf
- id: ev_secinfer_ia_dh_base
  defence_id: secinfer_l31
  benchmark: InjecAgent, Direct Harm, Base Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
- id: ev_none_ia_ds_base
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: InjecAgent, Data Stealing, Base Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  baseline_kind: off_the_shelf
- id: ev_secinfer_ia_ds_base
  defence_id: secinfer_l31
  benchmark: InjecAgent, Data Stealing, Base Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
- id: ev_none_ia_dh_enh
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: InjecAgent, Direct Harm, Enhanced Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  baseline_kind: off_the_shelf
- id: ev_secinfer_ia_dh_enh
  defence_id: secinfer_l31
  benchmark: InjecAgent, Direct Harm, Enhanced Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
- id: ev_none_ia_ds_enh
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: InjecAgent, Data Stealing, Enhanced Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  baseline_kind: off_the_shelf
- id: ev_secinfer_ia_ds_enh
  defence_id: secinfer_l31
  benchmark: InjecAgent, Data Stealing, Enhanced Attack
  attacker_type: static
  attacker_access: black_box
  judge: attacker-desired action predicted
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
- id: ev_none_dojo_clean
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  baseline_kind: off_the_shelf
- id: ev_none_dojo_atk
  defence_id: none:LLaMA3.1-8B-Instruct
  benchmark: AgentDojo with injections in tool results
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo utility and security checks
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
  baseline_kind: off_the_shelf
- id: ev_secinfer_dojo_clean
  defence_id: secinfer_l31
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
- id: ev_secinfer_dojo_atk
  defence_id: secinfer_l31
  benchmark: AgentDojo with injections in tool results
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo utility and security checks
  layout_same_as_defence: 'yes'
  measured_by: secinfer_arxiv2025
  page: 16
  decoding: not reported for the baselines; SecInfer samples with temperature and top-k
results:
- id: r_none_agnews_u
  evaluation_id: ev_none_agnews_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, AG News
  value: '0.79'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_agnews_ua
  evaluation_id: ev_none_agnews_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.09'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_agnews_asr
  evaluation_id: ev_none_agnews_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.64'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_mmlu_u
  evaluation_id: ev_none_mmlu_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, MMLU
  value: '0.41'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_mmlu_ua
  evaluation_id: ev_none_mmlu_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.03'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_mmlu_asr
  evaluation_id: ev_none_mmlu_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.54'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_gpqa_u
  evaluation_id: ev_none_gpqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, GPQA
  value: '0.26'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_gpqa_ua
  evaluation_id: ev_none_gpqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.03'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_gpqa_asr
  evaluation_id: ev_none_gpqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.55'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_narrqa_u
  evaluation_id: ev_none_narrqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, NarrativeQA
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_narrqa_ua
  evaluation_id: ev_none_narrqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.05'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_narrqa_asr
  evaluation_id: ev_none_narrqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.76'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_squad_u
  evaluation_id: ev_none_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SQuAD_v2
  value: '0.68'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_squad_ua
  evaluation_id: ev_none_squad_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.06'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_squad_asr
  evaluation_id: ev_none_squad_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.85'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_simpleqa_u
  evaluation_id: ev_none_simpleqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SimpleQA
  value: '0.62'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_simpleqa_ua
  evaluation_id: ev_none_simpleqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.15'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_simpleqa_asr
  evaluation_id: ev_none_simpleqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.87'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_agnews_u
  evaluation_id: ev_delim_agnews_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, AG News
  value: '0.78'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_agnews_ua
  evaluation_id: ev_delim_agnews_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.32'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_agnews_asr
  evaluation_id: ev_delim_agnews_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.54'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_mmlu_u
  evaluation_id: ev_delim_mmlu_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, MMLU
  value: '0.36'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_mmlu_ua
  evaluation_id: ev_delim_mmlu_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.19'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_mmlu_asr
  evaluation_id: ev_delim_mmlu_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_gpqa_u
  evaluation_id: ev_delim_gpqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, GPQA
  value: '0.22'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_gpqa_ua
  evaluation_id: ev_delim_gpqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.10'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_gpqa_asr
  evaluation_id: ev_delim_gpqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.53'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_narrqa_u
  evaluation_id: ev_delim_narrqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, NarrativeQA
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_narrqa_ua
  evaluation_id: ev_delim_narrqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.07'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_narrqa_asr
  evaluation_id: ev_delim_narrqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.72'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_squad_u
  evaluation_id: ev_delim_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SQuAD_v2
  value: '0.74'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_squad_ua
  evaluation_id: ev_delim_squad_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.18'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_squad_asr
  evaluation_id: ev_delim_squad_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.64'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_simpleqa_u
  evaluation_id: ev_delim_simpleqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SimpleQA
  value: '0.38'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_simpleqa_ua
  evaluation_id: ev_delim_simpleqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_delim_simpleqa_asr
  evaluation_id: ev_delim_simpleqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.71'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_agnews_u
  evaluation_id: ev_sandwich_agnews_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, AG News
  value: '0.88'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_agnews_ua
  evaluation_id: ev_sandwich_agnews_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.24'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_agnews_asr
  evaluation_id: ev_sandwich_agnews_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.85'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_mmlu_u
  evaluation_id: ev_sandwich_mmlu_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, MMLU
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_mmlu_ua
  evaluation_id: ev_sandwich_mmlu_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.22'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_mmlu_asr
  evaluation_id: ev_sandwich_mmlu_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_gpqa_u
  evaluation_id: ev_sandwich_gpqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, GPQA
  value: '0.22'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_gpqa_ua
  evaluation_id: ev_sandwich_gpqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.05'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_gpqa_asr
  evaluation_id: ev_sandwich_gpqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_narrqa_u
  evaluation_id: ev_sandwich_narrqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, NarrativeQA
  value: '0.51'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_narrqa_ua
  evaluation_id: ev_sandwich_narrqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.06'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_narrqa_asr
  evaluation_id: ev_sandwich_narrqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.82'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_squad_u
  evaluation_id: ev_sandwich_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SQuAD_v2
  value: '0.78'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_squad_ua
  evaluation_id: ev_sandwich_squad_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.04'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_squad_asr
  evaluation_id: ev_sandwich_squad_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_simpleqa_u
  evaluation_id: ev_sandwich_simpleqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SimpleQA
  value: '0.27'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_simpleqa_ua
  evaluation_id: ev_sandwich_simpleqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.57'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_sandwich_simpleqa_asr
  evaluation_id: ev_sandwich_simpleqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.85'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_agnews_u
  evaluation_id: ev_instr_agnews_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, AG News
  value: '0.83'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_agnews_ua
  evaluation_id: ev_instr_agnews_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.04'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_agnews_asr
  evaluation_id: ev_instr_agnews_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_mmlu_u
  evaluation_id: ev_instr_mmlu_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, MMLU
  value: '0.43'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_mmlu_ua
  evaluation_id: ev_instr_mmlu_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.09'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_mmlu_asr
  evaluation_id: ev_instr_mmlu_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.49'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_gpqa_u
  evaluation_id: ev_instr_gpqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, GPQA
  value: '0.25'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_gpqa_ua
  evaluation_id: ev_instr_gpqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.02'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_gpqa_asr
  evaluation_id: ev_instr_gpqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_narrqa_u
  evaluation_id: ev_instr_narrqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, NarrativeQA
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_narrqa_ua
  evaluation_id: ev_instr_narrqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.03'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_narrqa_asr
  evaluation_id: ev_instr_narrqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_squad_u
  evaluation_id: ev_instr_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SQuAD_v2
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_squad_ua
  evaluation_id: ev_instr_squad_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.03'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_squad_asr
  evaluation_id: ev_instr_squad_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.81'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_simpleqa_u
  evaluation_id: ev_instr_simpleqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SimpleQA
  value: '0.19'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_simpleqa_ua
  evaluation_id: ev_instr_simpleqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.21'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_instr_simpleqa_asr
  evaluation_id: ev_instr_simpleqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.80'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_agnews_u
  evaluation_id: ev_msa_agnews_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, AG News
  value: '0.85'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_agnews_ua
  evaluation_id: ev_msa_agnews_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.84'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_agnews_asr
  evaluation_id: ev_msa_agnews_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_mmlu_u
  evaluation_id: ev_msa_mmlu_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, MMLU
  value: '0.50'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_mmlu_ua
  evaluation_id: ev_msa_mmlu_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.49'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_mmlu_asr
  evaluation_id: ev_msa_mmlu_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_gpqa_u
  evaluation_id: ev_msa_gpqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, GPQA
  value: '0.29'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_gpqa_ua
  evaluation_id: ev_msa_gpqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.29'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_gpqa_asr
  evaluation_id: ev_msa_gpqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_narrqa_u
  evaluation_id: ev_msa_narrqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, NarrativeQA
  value: '0.53'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_narrqa_ua
  evaluation_id: ev_msa_narrqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.52'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_narrqa_asr
  evaluation_id: ev_msa_narrqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_squad_u
  evaluation_id: ev_msa_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SQuAD_v2
  value: '0.73'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_squad_ua
  evaluation_id: ev_msa_squad_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.67'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_squad_asr
  evaluation_id: ev_msa_squad_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_simpleqa_u
  evaluation_id: ev_msa_simpleqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SimpleQA
  value: '0.64'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_simpleqa_ua
  evaluation_id: ev_msa_simpleqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.53'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_simpleqa_asr
  evaluation_id: ev_msa_simpleqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_agnews_u
  evaluation_id: ev_secinfer_agnews_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, AG News
  value: '0.86'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_agnews_ua
  evaluation_id: ev_secinfer_agnews_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.81'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_agnews_asr
  evaluation_id: ev_secinfer_agnews_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_mmlu_u
  evaluation_id: ev_secinfer_mmlu_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, MMLU
  value: '0.49'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_mmlu_ua
  evaluation_id: ev_secinfer_mmlu_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.49'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_mmlu_asr
  evaluation_id: ev_secinfer_mmlu_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_gpqa_u
  evaluation_id: ev_secinfer_gpqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, GPQA
  value: '0.29'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_gpqa_ua
  evaluation_id: ev_secinfer_gpqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.29'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_gpqa_asr
  evaluation_id: ev_secinfer_gpqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_narrqa_u
  evaluation_id: ev_secinfer_narrqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, NarrativeQA
  value: '0.52'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_narrqa_ua
  evaluation_id: ev_secinfer_narrqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.48'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_narrqa_asr
  evaluation_id: ev_secinfer_narrqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_squad_u
  evaluation_id: ev_secinfer_squad_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SQuAD_v2
  value: '0.72'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_squad_ua
  evaluation_id: ev_secinfer_squad_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.64'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_squad_asr
  evaluation_id: ev_secinfer_squad_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_simpleqa_u
  evaluation_id: ev_secinfer_simpleqa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U, SimpleQA
  value: '0.59'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_simpleqa_ua
  evaluation_id: ev_secinfer_simpleqa_ca
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.53'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_simpleqa_asr
  evaluation_id: ev_secinfer_simpleqa_ca
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_agnews_ua_gcg
  evaluation_id: ev_msa_agnews_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.01'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_agnews_asr_gcg
  evaluation_id: ev_msa_agnews_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.63'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_mmlu_ua_gcg
  evaluation_id: ev_msa_mmlu_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.01'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_mmlu_asr_gcg
  evaluation_id: ev_msa_mmlu_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.54'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_gpqa_ua_gcg
  evaluation_id: ev_msa_gpqa_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.12'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_gpqa_asr_gcg
  evaluation_id: ev_msa_gpqa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_narrqa_ua_gcg
  evaluation_id: ev_msa_narrqa_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.01'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_narrqa_asr_gcg
  evaluation_id: ev_msa_narrqa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.47'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_squad_ua_gcg
  evaluation_id: ev_msa_squad_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.00'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_squad_asr_gcg
  evaluation_id: ev_msa_squad_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.25'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_simpleqa_ua_gcg
  evaluation_id: ev_msa_simpleqa_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.01'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_msa_simpleqa_asr_gcg
  evaluation_id: ev_msa_simpleqa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.37'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_agnews_ua_gcg
  evaluation_id: ev_secinfer_agnews_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, AG News
  value: '0.67'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_agnews_asr_gcg
  evaluation_id: ev_secinfer_agnews_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, AG News
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_mmlu_ua_gcg
  evaluation_id: ev_secinfer_mmlu_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, MMLU
  value: '0.44'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_mmlu_asr_gcg
  evaluation_id: ev_secinfer_mmlu_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, MMLU
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_gpqa_ua_gcg
  evaluation_id: ev_secinfer_gpqa_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, GPQA
  value: '0.25'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_gpqa_asr_gcg
  evaluation_id: ev_secinfer_gpqa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GPQA
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_narrqa_ua_gcg
  evaluation_id: ev_secinfer_narrqa_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, NarrativeQA
  value: '0.42'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_narrqa_asr_gcg
  evaluation_id: ev_secinfer_narrqa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, NarrativeQA
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_squad_ua_gcg
  evaluation_id: ev_secinfer_squad_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SQuAD_v2
  value: '0.34'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_squad_asr_gcg
  evaluation_id: ev_secinfer_squad_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SQuAD_v2
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_simpleqa_ua_gcg
  evaluation_id: ev_secinfer_simpleqa_gcg
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA, SimpleQA
  value: '0.31'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_secinfer_simpleqa_asr_gcg
  evaluation_id: ev_secinfer_simpleqa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SimpleQA
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 2
- id: r_none_ia_dh_base
  evaluation_id: ev_none_ia_dh_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Direct Harm, Base Attack
  value: '0.37'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_secinfer_ia_dh_base
  evaluation_id: ev_secinfer_ia_dh_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Direct Harm, Base Attack
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_none_ia_ds_base
  evaluation_id: ev_none_ia_ds_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Data Stealing, Base Attack
  value: '0.40'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_secinfer_ia_ds_base
  evaluation_id: ev_secinfer_ia_ds_base
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Data Stealing, Base Attack
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_none_ia_dh_enh
  evaluation_id: ev_none_ia_dh_enh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Direct Harm, Enhanced Attack
  value: '0.61'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_secinfer_ia_dh_enh
  evaluation_id: ev_secinfer_ia_dh_enh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Direct Harm, Enhanced Attack
  value: '0.01'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_none_ia_ds_enh
  evaluation_id: ev_none_ia_ds_enh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Data Stealing, Enhanced Attack
  value: '0.68'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_secinfer_ia_ds_enh
  evaluation_id: ev_secinfer_ia_ds_enh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Data Stealing, Enhanced Attack
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 5
- id: r_none_dojo_u
  evaluation_id: ev_none_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U
  value: '0.40'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_none_dojo_ua
  evaluation_id: ev_none_dojo_atk
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA
  value: '0.34'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_none_dojo_asr
  evaluation_id: ev_none_dojo_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.21'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_secinfer_dojo_u
  evaluation_id: ev_secinfer_dojo_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: U
  value: '0.39'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_secinfer_dojo_ua
  evaluation_id: ev_secinfer_dojo_atk
  quantity: R2attack
  coverage: direct
  metric_as_printed: UA
  value: '0.36'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
- id: r_secinfer_dojo_asr
  evaluation_id: ev_secinfer_dojo_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Table 6
comparisons:
- id: c_agnews_ca_si_none
  left: ev_secinfer_agnews_ca
  right: ev_none_agnews_ca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_agnews_clean_si_none
  left: ev_secinfer_agnews_clean
  right: ev_none_agnews_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_agnews_ca_si_msa
  left: ev_secinfer_agnews_ca
  right: ev_msa_agnews_ca
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign
  page: 15
- id: c_agnews_gcg_si_msa
  left: ev_secinfer_agnews_gcg
  right: ev_msa_agnews_gcg
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign under GCG
  page: 15
- id: c_mmlu_ca_si_none
  left: ev_secinfer_mmlu_ca
  right: ev_none_mmlu_ca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_mmlu_clean_si_none
  left: ev_secinfer_mmlu_clean
  right: ev_none_mmlu_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_mmlu_ca_si_msa
  left: ev_secinfer_mmlu_ca
  right: ev_msa_mmlu_ca
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign
  page: 15
- id: c_mmlu_gcg_si_msa
  left: ev_secinfer_mmlu_gcg
  right: ev_msa_mmlu_gcg
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign under GCG
  page: 15
- id: c_gpqa_ca_si_none
  left: ev_secinfer_gpqa_ca
  right: ev_none_gpqa_ca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_gpqa_clean_si_none
  left: ev_secinfer_gpqa_clean
  right: ev_none_gpqa_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_gpqa_ca_si_msa
  left: ev_secinfer_gpqa_ca
  right: ev_msa_gpqa_ca
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign
  page: 15
- id: c_gpqa_gcg_si_msa
  left: ev_secinfer_gpqa_gcg
  right: ev_msa_gpqa_gcg
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign under GCG
  page: 15
- id: c_narrqa_ca_si_none
  left: ev_secinfer_narrqa_ca
  right: ev_none_narrqa_ca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_narrqa_clean_si_none
  left: ev_secinfer_narrqa_clean
  right: ev_none_narrqa_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_narrqa_ca_si_msa
  left: ev_secinfer_narrqa_ca
  right: ev_msa_narrqa_ca
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign
  page: 15
- id: c_narrqa_gcg_si_msa
  left: ev_secinfer_narrqa_gcg
  right: ev_msa_narrqa_gcg
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign under GCG
  page: 15
- id: c_squad_ca_si_none
  left: ev_secinfer_squad_ca
  right: ev_none_squad_ca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_squad_clean_si_none
  left: ev_secinfer_squad_clean
  right: ev_none_squad_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_squad_ca_si_msa
  left: ev_secinfer_squad_ca
  right: ev_msa_squad_ca
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign
  page: 15
- id: c_squad_gcg_si_msa
  left: ev_secinfer_squad_gcg
  right: ev_msa_squad_gcg
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign under GCG
  page: 15
- id: c_simpleqa_ca_si_none
  left: ev_secinfer_simpleqa_ca
  right: ev_none_simpleqa_ca
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_simpleqa_clean_si_none
  left: ev_secinfer_simpleqa_clean
  right: ev_none_simpleqa_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (its sampling of several responses is part of SecInfer; the harness decoding is not stated)
  page: 15
- id: c_simpleqa_ca_si_msa
  left: ev_secinfer_simpleqa_ca
  right: ev_msa_simpleqa_ca
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign
  page: 15
- id: c_simpleqa_gcg_si_msa
  left: ev_secinfer_simpleqa_gcg
  right: ev_msa_simpleqa_gcg
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: SecInfer against MetaSecAlign under GCG
  page: 15
- id: c_ia_dh_base
  left: ev_secinfer_ia_dh_base
  right: ev_none_ia_dh_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_ia_ds_base
  left: ev_secinfer_ia_ds_base
  right: ev_none_ia_ds_base
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_ia_dh_enh
  left: ev_secinfer_ia_dh_enh
  right: ev_none_ia_dh_enh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_ia_ds_enh
  left: ev_secinfer_ia_ds_enh
  right: ev_none_ia_ds_enh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dojo_clean
  left: ev_secinfer_dojo_clean
  right: ev_none_dojo_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_dojo_atk
  left: ev_secinfer_dojo_atk
  right: ev_none_dojo_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 16
claims:
- id: k1
  text: SecInfer mitigates existing and adaptive attacks and outperforms state-of-the-art defences
  location: abstract
  anchor: SecInfer effectively mitigates both existing and adap- tive prompt injection attacks
  jobs:
  - R1
  supported_by:
  - c_agnews_ca_si_none
  - c_mmlu_ca_si_none
  - c_gpqa_ca_si_none
  - c_narrqa_ca_si_none
  - c_squad_ca_si_none
  - c_simpleqa_ca_si_none
  - c_agnews_gcg_si_msa
  - c_mmlu_gcg_si_msa
  - c_gpqa_gcg_si_msa
  - c_narrqa_gcg_si_msa
  - c_squad_gcg_si_msa
  - c_simpleqa_gcg_si_msa
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 2, 5, 6
    R2clean: Tables 2 and 6 (U)
    R2attack: Tables 2 and 6 (UA)

`rlhammer_arxiv2025` (2510.04885)

In [ ]:
%%writefile reports/records/rlhammer_arxiv2025.yaml
report:
  id: rlhammer_arxiv2025
  slug: '2510.04885'
  title: 'RL Is a Hammer and LLMs Are Nails: A Simple Reinforcement Learning Recipe for Strong Prompt Injection'
  first_author: Yuxin Wen
  venue: arXiv preprint (under review)
  peer_reviewed: 'no'
  arxiv: '2510.04885'
  version_read: arXiv v1
  first_date: '2025-10-06'
  read: F
  role: re_measures
  code_release: not checked
  extraction_scope: 'Sections 4.1 to 4.2: the Llama-3.1-8B-Instruct and Meta-SecAlign-8B columns of Table 1 (baseline attacks, GCG, and RL-Hammer trained against each target); the abstract claim (commercial models with built-in
    defences, which have no undefended counterpart here, Meta-SecAlign-70B, whose base differs, and Tables 3 to 7 not coded)'
defences:
- id: msa8b
  name: Meta-SecAlign-8B (released model, re-measured)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  name_as_printed: Meta-SecAlign-8B
  layout: tool use enabled by prompting (white-box models)
  anchor: consider Llama-3.1-8B-Instruct (Grattafiori et al., 2024), Meta-SecAlign-8B
evaluations:
- id: ev_none_default
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: static
  attacker_access: black_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: InjecAgent default injection
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
  baseline_kind: off_the_shelf
- id: ev_msa_default
  defence_id: msa8b
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: static
  attacker_access: black_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: InjecAgent default injection
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
- id: ev_none_enh
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: static
  attacker_access: black_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: InjecAgent enhanced injection
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
  baseline_kind: off_the_shelf
- id: ev_msa_enh
  defence_id: msa8b
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: static
  attacker_access: black_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: InjecAgent enhanced injection
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
- id: ev_none_gcg
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: optimisation
  attacker_access: white_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: GCG run directly on each white-box target
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
  baseline_kind: off_the_shelf
- id: ev_msa_gcg
  defence_id: msa8b
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: optimisation
  attacker_access: white_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: GCG run directly on each white-box target
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
- id: ev_none_rl
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: rl
  attacker_access: black_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: attacker model trained by RL against this target (with Llama-3.1-8B-Instruct as the easy training target)
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
  baseline_kind: off_the_shelf
- id: ev_msa_rl
  defence_id: msa8b
  benchmark: InjecAgent, 100 held-out test samples
  attacker_type: rl
  attacker_access: black_box
  judge: attacker's target tool call made (InjecAgent)
  layout_same_as_defence: 'yes'
  measured_by: rlhammer_arxiv2025
  page: 6
  attacker_budget: attacker model trained by RL against this target (with Llama-3.1-8B-Instruct as the easy training target)
  sample_size: '100'
  decoding: not reported
  pages:
    benchmark: 5
    sample_size: 5
results:
- id: r_none_default
  evaluation_id: ev_none_default
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Default Prompt
  value: '31.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_msa_default
  evaluation_id: ev_msa_default
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Default Prompt
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_none_enh
  evaluation_id: ev_none_enh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Enhanced
  value: '46.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_msa_enh
  evaluation_id: ev_msa_enh
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Enhanced
  value: '3.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_none_gcg
  evaluation_id: ev_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '32.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_msa_gcg
  evaluation_id: ev_msa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '21.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_none_rl
  evaluation_id: ev_none_rl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, RL-Hammer
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
- id: r_msa_rl
  evaluation_id: ev_msa_rl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, RL-Hammer
  value: '98.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 1
comparisons:
- id: c_default
  left: ev_msa_default
  right: ev_none_default
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_enh
  left: ev_msa_enh
  right: ev_none_enh
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gcg
  left: ev_msa_gcg
  right: ev_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_rl
  left: ev_msa_rl
  right: ev_none_rl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
claims:
- id: k1
  text: RL-Hammer reaches 98% ASR against GPT-4o and 72% against GPT-5 with the Instruction Hierarchy defence
  location: abstract
  anchor: RL-Hammer reaches a 98% ASR against GPT-4o
  jobs:
  - R1
  supported_by:
  - c_rl
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (InjecAgent)

`arlas_arxiv2025` (2510.05442)

In [ ]:
%%writefile reports/records/arlas_arxiv2025.yaml
report:
  id: arlas_arxiv2025
  slug: '2510.05442'
  title: Adversarial Reinforcement Learning for Large Language Model Agent Safety
  first_author: Zizhao Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2510.05442'
  version_read: arXiv v1
  first_date: '2025-10-06'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Sections 3 and 4.2: Table 1 (AgentDojo, base model, ARLAS and the two ablations, both base models); the abstract claim (BrowserGym results in Figures 3 and 4, chart only, the SPAG and ART baselines and
    the diversity analysis not coded)'
defences:
- id: arlas_g12
  name: ARLAS
  origin: this_report
  base_checkpoint: Gemma3 12B
  axis1:
  - no_signal
  axis2:
  - sft
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  components:
  - an attacker LLM and the agent LLM co-trained by RL as a two-player zero-sum game
  - 'population-based learning: the agent trains against all earlier attacker checkpoints'
  - a supervised fine-tuning stage before RL
  layout: agent loop with tool outputs carrying injections
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: ARLAS (ours)
  pages:
    components: 1
- id: arlas_q14
  name: ARLAS
  origin: this_report
  base_checkpoint: Qwen3 14B
  axis1:
  - no_signal
  axis2:
  - sft
  - adversarial_rl
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  components:
  - an attacker LLM and the agent LLM co-trained by RL as a two-player zero-sum game
  - 'population-based learning: the agent trains against all earlier attacker checkpoints'
  - a supervised fine-tuning stage before RL
  layout: agent loop with tool outputs carrying injections
  layout_compat: *id001
  anchor: ARLAS (ours)
  pages:
    components: 1
- id: arlas_nopbl_g12
  name: ARLAS without population-based learning
  origin: this_report
  base_checkpoint: Gemma3 12B
  axis1:
  - no_signal
  axis2:
  - sft
  - adversarial_rl
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: ARLAS w/o PBL
- id: arlas_noal_g12
  name: ARLAS without adversarial learning
  origin: this_report
  base_checkpoint: Gemma3 12B
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: ARLAS w/o AL
evaluations:
- id: ev_g12_none
  defence_id: none:Gemma3 12B
  benchmark: AgentDojo (out-of-distribution tasks and attack patterns)
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checks: attack success and task success'
  layout_same_as_defence: 'yes'
  measured_by: arlas_arxiv2025
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_g12_arlas
  defence_id: arlas_g12
  benchmark: AgentDojo (out-of-distribution tasks and attack patterns)
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checks: attack success and task success'
  layout_same_as_defence: 'yes'
  measured_by: arlas_arxiv2025
  page: 8
  decoding: not reported
  run: mean of 3 random seeds
- id: ev_g12_nopbl
  defence_id: arlas_nopbl_g12
  benchmark: AgentDojo (out-of-distribution tasks and attack patterns)
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checks: attack success and task success'
  layout_same_as_defence: 'yes'
  measured_by: arlas_arxiv2025
  page: 8
  decoding: not reported
  run: mean of 3 random seeds
- id: ev_g12_noal
  defence_id: arlas_noal_g12
  benchmark: AgentDojo (out-of-distribution tasks and attack patterns)
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checks: attack success and task success'
  layout_same_as_defence: 'yes'
  measured_by: arlas_arxiv2025
  page: 8
  decoding: not reported
  run: mean of 3 random seeds
- id: ev_q14_none
  defence_id: none:Qwen3 14B
  benchmark: AgentDojo (out-of-distribution tasks and attack patterns)
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checks: attack success and task success'
  layout_same_as_defence: 'yes'
  measured_by: arlas_arxiv2025
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q14_arlas
  defence_id: arlas_q14
  benchmark: AgentDojo (out-of-distribution tasks and attack patterns)
  attacker_type: static
  attacker_access: black_box
  judge: 'AgentDojo checks: attack success and task success'
  layout_same_as_defence: 'yes'
  measured_by: arlas_arxiv2025
  page: 8
  decoding: not reported
  run: mean of 3 random seeds
results:
- id: r_g12_none_asr
  evaluation_id: ev_g12_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate (%)
  value: '6.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_g12_none_tsr
  evaluation_id: ev_g12_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Task Success Rate (%)
  value: '24.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_g12_arlas_asr
  evaluation_id: ev_g12_arlas
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate (%)
  value: '5.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±0.1, 3 seeds
- id: r_g12_arlas_tsr
  evaluation_id: ev_g12_arlas
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Task Success Rate (%)
  value: '25.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±0.1, 3 seeds
- id: r_g12_nopbl_asr
  evaluation_id: ev_g12_nopbl
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate (%)
  value: '6.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±0.1, 3 seeds
- id: r_g12_nopbl_tsr
  evaluation_id: ev_g12_nopbl
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Task Success Rate (%)
  value: '23.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±1.5, 3 seeds
- id: r_g12_noal_asr
  evaluation_id: ev_g12_noal
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate (%)
  value: '8.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±0.0, 3 seeds
- id: r_g12_noal_tsr
  evaluation_id: ev_g12_noal
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Task Success Rate (%)
  value: '21.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±1.3, 3 seeds
- id: r_q14_none_asr
  evaluation_id: ev_q14_none
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate (%)
  value: '1.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q14_none_tsr
  evaluation_id: ev_q14_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Task Success Rate (%)
  value: '30.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q14_arlas_asr
  evaluation_id: ev_q14_arlas
  quantity: R1
  coverage: direct
  metric_as_printed: Attack Success Rate (%)
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±0.1, 3 seeds
- id: r_q14_arlas_tsr
  evaluation_id: ev_q14_arlas
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Task Success Rate (%)
  value: '31.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
  spread: standard error ±0.8, 3 seeds
comparisons:
- id: c_g12
  left: ev_g12_arlas
  right: ev_g12_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q14
  left: ev_q14_arlas
  right: ev_q14_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_g12_pbl
  left: ev_g12_arlas
  right: ev_g12_nopbl
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: population-based learning (on against off)
  page: 8
- id: c_g12_al
  left: ev_g12_arlas
  right: ev_g12_noal
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: adversarial learning (on against task-only training)
  page: 8
claims:
- id: k1
  text: ARLAS lowers attack success and raises task success
  location: abstract
  anchor: agents fine-tuned with ARLAS achieve a sig- nificantly lower attack success rate
  jobs:
  - R1
  - R2
  supported_by:
  - c_g12
  - c_q14
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: unclear
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (AgentDojo)
    R2attack: Table 1 task success rate; whether it is measured on attacked runs is not stated

`attackermovessecond_arxiv2025` (2510.09023)

In [ ]:
%%writefile reports/records/attackermovessecond_arxiv2025.yaml
report:
  id: attackermovessecond_arxiv2025
  slug: '2510.09023'
  title: 'The Attacker Moves Second: Stronger Adaptive Attacks Bypass Defenses against LLM Jailbreaks and Prompt Injections'
  first_author: Milad Nasr
  venue: arXiv preprint (under review)
  peer_reviewed: 'no'
  arxiv: '2510.09023'
  version_read: arXiv v1
  first_date: '2025-10-10'
  read: F
  role: re_measures
  code_release: not checked
  extraction_scope: 'Sections 3 to 5 and Appendix C: Table 7 rows None, Spotlighting and Prompt Sandwiching for all four models (MetaSecAlign 70B is coded as the defended model against Llama-3.3 70B); the abstract claim (detectors,
    Circuit Breakers, the RL and gradient attacks on StruQ and other trained defences, the human red-teaming counts and Tables 2 to 6 not coded)'
defences:
- id: msa70b
  name: MetaSecAlign 70B (released model, re-measured)
  origin: other_report
  base_checkpoint: Llama-3.3 70B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  name_as_printed: MetaSecAlign 70B
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: We re-evaluated this defense in the same agentic setting using our search-based attack
- id: spot_gem
  name: Spotlighting
  origin: other_report
  base_checkpoint: Gemini-2.5 Pro
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Spotlighting marks trusted text with special delimiter tokens
- id: sand_gem
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Gemini-2.5 Pro
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: we focus on Spotlighting (Hines et al., 2024), Prompt Sandwiching
- id: spot_g5m
  name: Spotlighting
  origin: other_report
  base_checkpoint: GPT-5 Mini
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Spotlighting marks trusted text with special delimiter tokens
- id: sand_g5m
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: GPT-5 Mini
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: we focus on Spotlighting (Hines et al., 2024), Prompt Sandwiching
- id: spot_l33
  name: Spotlighting
  origin: other_report
  base_checkpoint: Llama-3.3 70B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Spotlighting marks trusted text with special delimiter tokens
- id: sand_l33
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: Llama-3.3 70B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: we focus on Spotlighting (Hines et al., 2024), Prompt Sandwiching
- id: spot_msa
  name: Spotlighting
  origin: other_report
  base_checkpoint: MetaSecAlign 70B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: Spotlighting marks trusted text with special delimiter tokens
- id: sand_msa
  name: Prompt Sandwiching
  origin: other_report
  base_checkpoint: MetaSecAlign 70B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: AgentDojo agent loop; tool outputs carry the injection
  layout_compat: *id001
  anchor: we focus on Spotlighting (Hines et al., 2024), Prompt Sandwiching
evaluations:
- id: ev_none_gem_clean
  defence_id: none:Gemini-2.5 Pro
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_none_gem_static
  defence_id: none:Gemini-2.5 Pro
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_none_gem_search
  defence_id: none:Gemini-2.5 Pro
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
  baseline_kind: off_the_shelf
- id: ev_none_g5m_clean
  defence_id: none:GPT-5 Mini
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_none_g5m_static
  defence_id: none:GPT-5 Mini
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_none_g5m_search
  defence_id: none:GPT-5 Mini
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
  baseline_kind: off_the_shelf
- id: ev_none_l33_clean
  defence_id: none:Llama-3.3 70B
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_none_l33_static
  defence_id: none:Llama-3.3 70B
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_none_l33_search
  defence_id: none:Llama-3.3 70B
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
  baseline_kind: off_the_shelf
- id: ev_none_msa_clean
  defence_id: msa70b
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_none_msa_static
  defence_id: msa70b
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_none_msa_search
  defence_id: msa70b
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_spot_gem_clean
  defence_id: spot_gem
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_spot_gem_static
  defence_id: spot_gem
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_spot_gem_search
  defence_id: spot_gem
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_spot_g5m_clean
  defence_id: spot_g5m
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_spot_g5m_static
  defence_id: spot_g5m
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_spot_g5m_search
  defence_id: spot_g5m
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_spot_l33_clean
  defence_id: spot_l33
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_spot_l33_static
  defence_id: spot_l33
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_spot_l33_search
  defence_id: spot_l33
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_spot_msa_clean
  defence_id: spot_msa
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_spot_msa_static
  defence_id: spot_msa
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_spot_msa_search
  defence_id: spot_msa
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_sand_gem_clean
  defence_id: sand_gem
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_sand_gem_static
  defence_id: sand_gem
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_sand_gem_search
  defence_id: sand_gem
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_sand_g5m_clean
  defence_id: sand_g5m
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_sand_g5m_static
  defence_id: sand_g5m
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_sand_g5m_search
  defence_id: sand_g5m
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_sand_l33_clean
  defence_id: sand_l33
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_sand_l33_static
  defence_id: sand_l33
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_sand_l33_search
  defence_id: sand_l33
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
- id: ev_sand_msa_clean
  defence_id: sand_msa
  benchmark: AgentDojo user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo utility check
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  decoding: not reported
- id: ev_sand_msa_static
  defence_id: sand_msa
  benchmark: AgentDojo with the static important-instructions template
  attacker_type: static
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: AgentDojo important-instructions template
  decoding: not reported
- id: ev_sand_msa_search
  defence_id: sand_msa
  benchmark: AgentDojo scenarios attacked by search
  attacker_type: adaptive
  attacker_access: black_box
  judge: attacker goal met (AgentDojo security check)
  layout_same_as_defence: 'yes'
  measured_by: attackermovessecond_arxiv2025
  page: 25
  attacker_budget: adaptive search attack (LLM-guided), up to 800 queries per scenario
  decoding: not reported
  pages:
    attacker_budget: 25
results:
- id: r_none_gem_u
  evaluation_id: ev_none_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '74.23'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_gem_static
  evaluation_id: ev_none_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_gem_search
  evaluation_id: ev_none_gem_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_g5m_u
  evaluation_id: ev_none_g5m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '72.16'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_g5m_static
  evaluation_id: ev_none_g5m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_g5m_search
  evaluation_id: ev_none_g5m_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_l33_u
  evaluation_id: ev_none_l33_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '56.70'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_l33_static
  evaluation_id: ev_none_l33_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_l33_search
  evaluation_id: ev_none_l33_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_msa_u
  evaluation_id: ev_none_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '73.20'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_msa_static
  evaluation_id: ev_none_msa_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_none_msa_search
  evaluation_id: ev_none_msa_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_gem_u
  evaluation_id: ev_spot_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '75.26'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_gem_static
  evaluation_id: ev_spot_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_gem_search
  evaluation_id: ev_spot_gem_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_g5m_u
  evaluation_id: ev_spot_g5m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '76.29'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_g5m_static
  evaluation_id: ev_spot_g5m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_g5m_search
  evaluation_id: ev_spot_g5m_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_l33_u
  evaluation_id: ev_spot_l33_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '39.18'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_l33_static
  evaluation_id: ev_spot_l33_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_l33_search
  evaluation_id: ev_spot_l33_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_msa_u
  evaluation_id: ev_spot_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '68.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_msa_static
  evaluation_id: ev_spot_msa_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_spot_msa_search
  evaluation_id: ev_spot_msa_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_gem_u
  evaluation_id: ev_sand_gem_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '73.20'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_gem_static
  evaluation_id: ev_sand_gem_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '21'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_gem_search
  evaluation_id: ev_sand_gem_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_g5m_u
  evaluation_id: ev_sand_g5m_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '75.26'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_g5m_static
  evaluation_id: ev_sand_g5m_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_g5m_search
  evaluation_id: ev_sand_g5m_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_l33_u
  evaluation_id: ev_sand_l33_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '53.61'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_l33_static
  evaluation_id: ev_sand_l33_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_l33_search
  evaluation_id: ev_sand_l33_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_msa_u
  evaluation_id: ev_sand_msa_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility
  value: '79.38'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_msa_static
  evaluation_id: ev_sand_msa_static
  quantity: R1
  coverage: direct
  metric_as_printed: Static ASR
  value: '4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
- id: r_sand_msa_search
  evaluation_id: ev_sand_msa_search
  quantity: R1
  coverage: direct
  metric_as_printed: Search Attack ASR
  value: '99'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 25
  location: Table 7
comparisons:
- id: c_spot_gem_clean
  left: ev_spot_gem_clean
  right: ev_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_gem_static
  left: ev_spot_gem_static
  right: ev_none_gem_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_gem_search
  left: ev_spot_gem_search
  right: ev_none_gem_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_gem_clean
  left: ev_sand_gem_clean
  right: ev_none_gem_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_gem_static
  left: ev_sand_gem_static
  right: ev_none_gem_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_gem_search
  left: ev_sand_gem_search
  right: ev_none_gem_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_g5m_clean
  left: ev_spot_g5m_clean
  right: ev_none_g5m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_g5m_static
  left: ev_spot_g5m_static
  right: ev_none_g5m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_g5m_search
  left: ev_spot_g5m_search
  right: ev_none_g5m_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_g5m_clean
  left: ev_sand_g5m_clean
  right: ev_none_g5m_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_g5m_static
  left: ev_sand_g5m_static
  right: ev_none_g5m_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_g5m_search
  left: ev_sand_g5m_search
  right: ev_none_g5m_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_l33_clean
  left: ev_spot_l33_clean
  right: ev_none_l33_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_l33_static
  left: ev_spot_l33_static
  right: ev_none_l33_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_l33_search
  left: ev_spot_l33_search
  right: ev_none_l33_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_l33_clean
  left: ev_sand_l33_clean
  right: ev_none_l33_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_l33_static
  left: ev_sand_l33_static
  right: ev_none_l33_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_l33_search
  left: ev_sand_l33_search
  right: ev_none_l33_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_msa_clean
  left: ev_spot_msa_clean
  right: ev_none_msa_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_msa_static
  left: ev_spot_msa_static
  right: ev_none_msa_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_spot_msa_search
  left: ev_spot_msa_search
  right: ev_none_msa_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_msa_clean
  left: ev_sand_msa_clean
  right: ev_none_msa_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_msa_static
  left: ev_sand_msa_static
  right: ev_none_msa_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_sand_msa_search
  left: ev_sand_msa_search
  right: ev_none_msa_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 25
- id: c_msa_vs_l33_clean
  left: ev_none_msa_clean
  right: ev_none_l33_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: MetaSecAlign 70B against its base Llama-3.3 70B
  page: 25
- id: c_msa_vs_l33_static
  left: ev_none_msa_static
  right: ev_none_l33_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: MetaSecAlign 70B against its base Llama-3.3 70B
  page: 25
- id: c_msa_vs_l33_search
  left: ev_none_msa_search
  right: ev_none_l33_search
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: MetaSecAlign 70B against its base Llama-3.3 70B
  page: 25
claims:
- id: k1
  text: adaptive attacks bypass 12 recent defences with attack success above 90% for most
  location: abstract
  anchor: we bypass 12 recent defenses (based on a diverse set of techniques)
  jobs:
  - R1
  supported_by:
  - c_spot_gem_search
  - c_spot_g5m_search
  - c_spot_l33_search
  - c_spot_msa_search
  - c_sand_gem_search
  - c_sand_g5m_search
  - c_sand_l33_search
  - c_sand_msa_search
  - c_msa_vs_l33_search
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 7 (static and search ASR)
    R2clean: Table 7 (utility)

`verih_arxiv2025` (2511.04694)

In [ ]:
%%writefile reports/records/verih_arxiv2025.yaml
report:
  id: verih_arxiv2025
  slug: '2511.04694'
  title: Reasoning Up the Instruction Ladder for Controllable Language Models
  first_author: Zishuo Zheng
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2511.04694'
  version_read: arXiv v5
  first_date: '2025-10-30'
  read: F
  role: proposes
  code_release: not checked
  later_version: Findings of ACL 2026, DOI 10.18653/v1/2026.findings-acl.1960 (from the search record; not read)
  extraction_scope: 'Sections 3 to 5: the TensorTrust inject and helpful columns of Table 3 for all five models, rows w/ CoT, w/ CoT+SysHint+GuardRules and +VerIH; the abstract claim (instruction-following and IHEval results,
    Harmbench and WildJailbreak, Tables 2 and 4 and the appendix not coded)'
defences:
- id: verih_q4
  name: VerIH reinforcement learning (instruction-hierarchy reasoning)
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 'RL with verifiable rewards on VerIH: constraint-following tasks with aligned and conflicting system and user instructions'
  - SysHint system prompt asking the model to reason about the system-user relationship, kept in training
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: &id001
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: +VerIH refers to our method that is trained with our dataset VerIH
  pages:
    isolation: 8
- id: guard_q4
  name: SysHint + GuardRules (prompting)
  origin: this_report
  base_checkpoint: Qwen3-4B
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: SysHint is a system prompt that explicitly prompts models
- id: verih_q8
  name: VerIH reinforcement learning (instruction-hierarchy reasoning)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 'RL with verifiable rewards on VerIH: constraint-following tasks with aligned and conflicting system and user instructions'
  - SysHint system prompt asking the model to reason about the system-user relationship, kept in training
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: +VerIH refers to our method that is trained with our dataset VerIH
  pages:
    isolation: 8
- id: guard_q8
  name: SysHint + GuardRules (prompting)
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: SysHint is a system prompt that explicitly prompts models
- id: verih_q14
  name: VerIH reinforcement learning (instruction-hierarchy reasoning)
  origin: this_report
  base_checkpoint: Qwen3-14B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 'RL with verifiable rewards on VerIH: constraint-following tasks with aligned and conflicting system and user instructions'
  - SysHint system prompt asking the model to reason about the system-user relationship, kept in training
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: +VerIH refers to our method that is trained with our dataset VerIH
  pages:
    isolation: 8
- id: guard_q14
  name: SysHint + GuardRules (prompting)
  origin: this_report
  base_checkpoint: Qwen3-14B
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: SysHint is a system prompt that explicitly prompts models
- id: verih_q36
  name: VerIH reinforcement learning (instruction-hierarchy reasoning)
  origin: this_report
  base_checkpoint: Qwen3.6-35B-A3B
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 'RL with verifiable rewards on VerIH: constraint-following tasks with aligned and conflicting system and user instructions'
  - SysHint system prompt asking the model to reason about the system-user relationship, kept in training
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: +VerIH refers to our method that is trained with our dataset VerIH
  pages:
    isolation: 8
- id: guard_q36
  name: SysHint + GuardRules (prompting)
  origin: this_report
  base_checkpoint: Qwen3.6-35B-A3B
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: SysHint is a system prompt that explicitly prompts models
- id: verih_phi
  name: VerIH reinforcement learning (instruction-hierarchy reasoning)
  origin: this_report
  base_checkpoint: Phi-4-mini-reasoning
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 'RL with verifiable rewards on VerIH: constraint-following tasks with aligned and conflicting system and user instructions'
  - SysHint system prompt asking the model to reason about the system-user relationship, kept in training
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: +VerIH refers to our method that is trained with our dataset VerIH
  pages:
    isolation: 8
- id: guard_phi
  name: SysHint + GuardRules (prompting)
  origin: this_report
  base_checkpoint: Phi-4-mini-reasoning
  axis1:
  - chat_roles
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt with the higher-priority instruction, user prompt with the request
  layout_compat: *id001
  anchor: SysHint is a system prompt that explicitly prompts models
evaluations:
- id: ev_q4_cot_inject
  defence_id: none:Qwen3-4B with thinking (w/ CoT)
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q4_cot_helpful
  defence_id: none:Qwen3-4B with thinking (w/ CoT)
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q4_guard_inject
  defence_id: guard_q4
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q4_guard_helpful
  defence_id: guard_q4
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q4_verih_inject
  defence_id: verih_q4
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q4_verih_helpful
  defence_id: verih_q4
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q8_cot_inject
  defence_id: none:Qwen3-8B with thinking (w/ CoT)
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q8_cot_helpful
  defence_id: none:Qwen3-8B with thinking (w/ CoT)
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q8_guard_inject
  defence_id: guard_q8
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q8_guard_helpful
  defence_id: guard_q8
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q8_verih_inject
  defence_id: verih_q8
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q8_verih_helpful
  defence_id: verih_q8
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q14_cot_inject
  defence_id: none:Qwen3-14B with thinking (w/ CoT)
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q14_cot_helpful
  defence_id: none:Qwen3-14B with thinking (w/ CoT)
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q14_guard_inject
  defence_id: guard_q14
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q14_guard_helpful
  defence_id: guard_q14
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q14_verih_inject
  defence_id: verih_q14
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q14_verih_helpful
  defence_id: verih_q14
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q36_cot_inject
  defence_id: none:Qwen3.6-35B-A3B with thinking (w/ CoT)
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q36_cot_helpful
  defence_id: none:Qwen3.6-35B-A3B with thinking (w/ CoT)
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_q36_guard_inject
  defence_id: guard_q36
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q36_guard_helpful
  defence_id: guard_q36
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_q36_verih_inject
  defence_id: verih_q36
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_q36_verih_helpful
  defence_id: verih_q36
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_phi_cot_inject
  defence_id: none:Phi-4-mini-reasoning with thinking (w/ CoT)
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_phi_cot_helpful
  defence_id: none:Phi-4-mini-reasoning with thinking (w/ CoT)
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_phi_guard_inject
  defence_id: guard_phi
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_phi_guard_helpful
  defence_id: guard_phi
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
- id: ev_phi_verih_inject
  defence_id: verih_phi
  benchmark: TensorTrust injection (hijacking) set
  attacker_type: human
  attacker_access: black_box
  judge: attack succeeds when the injected goal is met (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  attacker_budget: TensorTrust player-written attacks
  decoding: not reported
- id: ev_phi_verih_helpful
  defence_id: verih_phi
  benchmark: TensorTrust helpful set (benign requests)
  attacker_type: none
  attacker_access: not_applicable
  judge: request complied with (Appendix D)
  layout_same_as_defence: 'yes'
  measured_by: verih_arxiv2025
  page: 7
  decoding: not reported
results:
- id: r_q4_cot_inject
  evaluation_id: ev_q4_cot_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '59.49'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q4_cot_helpful
  evaluation_id: ev_q4_cot_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '82.74'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q4_guard_inject
  evaluation_id: ev_q4_guard_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '54.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q4_guard_helpful
  evaluation_id: ev_q4_guard_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '86.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q4_verih_inject
  evaluation_id: ev_q4_verih_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '46.77'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q4_verih_helpful
  evaluation_id: ev_q4_verih_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '86.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q8_cot_inject
  evaluation_id: ev_q8_cot_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '55.91'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q8_cot_helpful
  evaluation_id: ev_q8_cot_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '83.96'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q8_guard_inject
  evaluation_id: ev_q8_guard_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '49.13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q8_guard_helpful
  evaluation_id: ev_q8_guard_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '86.79'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q8_verih_inject
  evaluation_id: ev_q8_verih_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '32.58'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q8_verih_helpful
  evaluation_id: ev_q8_verih_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '86.79'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q14_cot_inject
  evaluation_id: ev_q14_cot_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '51.32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q14_cot_helpful
  evaluation_id: ev_q14_cot_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '86.23'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q14_guard_inject
  evaluation_id: ev_q14_guard_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '50.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q14_guard_helpful
  evaluation_id: ev_q14_guard_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '87.45'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q14_verih_inject
  evaluation_id: ev_q14_verih_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '36.63'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q14_verih_helpful
  evaluation_id: ev_q14_verih_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '86.79'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q36_cot_inject
  evaluation_id: ev_q36_cot_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '12.54'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q36_cot_helpful
  evaluation_id: ev_q36_cot_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '78.49'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q36_guard_inject
  evaluation_id: ev_q36_guard_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '11.77'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q36_guard_helpful
  evaluation_id: ev_q36_guard_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '81.04'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q36_verih_inject
  evaluation_id: ev_q36_verih_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '6.76'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q36_verih_helpful
  evaluation_id: ev_q36_verih_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '79.91'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_phi_cot_inject
  evaluation_id: ev_phi_cot_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '38.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_phi_cot_helpful
  evaluation_id: ev_phi_cot_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '31.32'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_phi_guard_inject
  evaluation_id: ev_phi_guard_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '39.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_phi_guard_helpful
  evaluation_id: ev_phi_guard_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '33.30'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_phi_verih_inject
  evaluation_id: ev_phi_verih_inject
  quantity: R1
  coverage: direct
  metric_as_printed: TensorTrust inject
  value: '57.93'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_phi_verih_helpful
  evaluation_id: ev_phi_verih_helpful
  quantity: R2clean
  coverage: proxy
  metric_as_printed: TensorTrust helpful
  value: '71.70'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
comparisons:
- id: c_q4_inject_verih_cot
  left: ev_q4_verih_inject
  right: ev_q4_cot_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q4_inject_verih_guard
  left: ev_q4_verih_inject
  right: ev_q4_guard_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q4_helpful_verih_cot
  left: ev_q4_verih_helpful
  right: ev_q4_cot_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q4_helpful_verih_guard
  left: ev_q4_verih_helpful
  right: ev_q4_guard_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q8_inject_verih_cot
  left: ev_q8_verih_inject
  right: ev_q8_cot_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q8_inject_verih_guard
  left: ev_q8_verih_inject
  right: ev_q8_guard_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q8_helpful_verih_cot
  left: ev_q8_verih_helpful
  right: ev_q8_cot_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q8_helpful_verih_guard
  left: ev_q8_verih_helpful
  right: ev_q8_guard_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q14_inject_verih_cot
  left: ev_q14_verih_inject
  right: ev_q14_cot_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q14_inject_verih_guard
  left: ev_q14_verih_inject
  right: ev_q14_guard_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q14_helpful_verih_cot
  left: ev_q14_verih_helpful
  right: ev_q14_cot_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q14_helpful_verih_guard
  left: ev_q14_verih_helpful
  right: ev_q14_guard_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q36_inject_verih_cot
  left: ev_q36_verih_inject
  right: ev_q36_cot_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q36_inject_verih_guard
  left: ev_q36_verih_inject
  right: ev_q36_guard_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_q36_helpful_verih_cot
  left: ev_q36_verih_helpful
  right: ev_q36_cot_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q36_helpful_verih_guard
  left: ev_q36_verih_helpful
  right: ev_q36_guard_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_phi_inject_verih_cot
  left: ev_phi_verih_inject
  right: ev_phi_cot_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_inject_verih_guard
  left: ev_phi_verih_inject
  right: ev_phi_guard_inject
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
- id: c_phi_helpful_verih_cot
  left: ev_phi_verih_helpful
  right: ev_phi_cot_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_helpful_verih_guard
  left: ev_phi_verih_helpful
  right: ev_phi_guard_helpful
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: VerIH training against the same model with the SysHint and GuardRules prompts
  page: 7
claims:
- id: k1
  text: more robust to jailbreak and prompt injection, absolute attack success down by up to 20%
  location: abstract
  anchor: reducing abso- lute attack success rates by up to 20%
  jobs:
  - R1
  supported_by:
  - c_q4_inject_verih_cot
  - c_q8_inject_verih_cot
  - c_q14_inject_verih_cot
  - c_q36_inject_verih_cot
  - c_phi_inject_verih_cot
  page: 1
  contradicted_by:
  - r_phi_verih_inject
coverage:
  R1: direct
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3 (TensorTrust inject)
    R2clean: Table 3 (TensorTrust helpful)
schema_gaps:
- where: defences verih_*
  note: VerIH trains with reinforcement learning from verifiable rewards; axis 2 has no value for non-adversarial RL, so it is coded preference_optimisation

`promptfencing_arxiv2025` (2511.19727)

In [ ]:
%%writefile reports/records/promptfencing_arxiv2025.yaml
report:
  id: promptfencing_arxiv2025
  slug: '2511.19727'
  title: 'Prompt Fencing: A Cryptographic Approach to Establishing Security Boundaries in Large Language Model Prompts'
  first_author: Steven Peh
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2511.19727'
  version_read: arXiv (version not printed on the PDF)
  first_date: '2025-11-24'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'Sections 5 to 6.3.1: Table 2 (average injection success counts, both models, baseline and fenced); the abstract claim (overhead in Table 1, the boundary forgery tests of Section 6.3.2 and Table 4 not coded)'
defences:
- id: fence_gem
  name: Prompt Fencing (simulated fence awareness)
  origin: this_report
  base_checkpoint: Gemini 2.5 Pro
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 24
  components:
  - 'segments decorated with HMAC-signed metadata: trust rating and content type'
  - fence signatures validated before the call
  - prompt instructions telling the model to treat untrusted fences as data
  layout: prompt segments wrapped in fences carrying signed metadata (trust rating, content type), with fence-awareness instructions
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: comparing baseline (unfenced) against fence-aware conditions
  pages:
    components: 1
- id: fence_cla
  name: Prompt Fencing (simulated fence awareness)
  origin: this_report
  base_checkpoint: Claude Sonnet 4.5
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: no_ablation
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 24
  components:
  - 'segments decorated with HMAC-signed metadata: trust rating and content type'
  - fence signatures validated before the call
  - prompt instructions telling the model to treat untrusted fences as data
  layout: prompt segments wrapped in fences carrying signed metadata (trust rating, content type), with fence-awareness instructions
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: comparing baseline (unfenced) against fence-aware conditions
  pages:
    components: 1
evaluations:
- id: ev_gem_base
  defence_id: none:Gemini 2.5 Pro
  benchmark: review-analysis task, 100 samples with injections in 50 (boundary-confusion markers and a fake system note)
  attacker_type: static
  attacker_access: black_box
  judge: injected output (Final Rating=100) produced
  layout_same_as_defence: 'yes'
  measured_by: promptfencing_arxiv2025
  page: 24
  sample_size: 50 injected samples per run
  run: average of 3 runs
  decoding: not reported
  pages:
    benchmark: 25
  baseline_kind: off_the_shelf
- id: ev_gem_fence
  defence_id: fence_gem
  benchmark: review-analysis task, 100 samples with injections in 50 (boundary-confusion markers and a fake system note)
  attacker_type: static
  attacker_access: black_box
  judge: injected output (Final Rating=100) produced
  layout_same_as_defence: 'yes'
  measured_by: promptfencing_arxiv2025
  page: 24
  sample_size: 50 injected samples per run
  run: average of 3 runs
  decoding: not reported
  pages:
    benchmark: 25
- id: ev_cla_base
  defence_id: none:Claude Sonnet 4.5
  benchmark: review-analysis task, 100 samples with injections in 50 (boundary-confusion markers and a fake system note)
  attacker_type: static
  attacker_access: black_box
  judge: injected output (Final Rating=100) produced
  layout_same_as_defence: 'yes'
  measured_by: promptfencing_arxiv2025
  page: 24
  sample_size: 50 injected samples per run
  run: average of 3 runs
  decoding: not reported
  pages:
    benchmark: 25
  baseline_kind: off_the_shelf
- id: ev_cla_fence
  defence_id: fence_cla
  benchmark: review-analysis task, 100 samples with injections in 50 (boundary-confusion markers and a fake system note)
  attacker_type: static
  attacker_access: black_box
  judge: injected output (Final Rating=100) produced
  layout_same_as_defence: 'yes'
  measured_by: promptfencing_arxiv2025
  page: 24
  sample_size: 50 injected samples per run
  run: average of 3 runs
  decoding: not reported
  pages:
    benchmark: 25
results:
- id: r_gem_base
  evaluation_id: ev_gem_base
  quantity: R1
  coverage: direct
  metric_as_printed: Average Success Count (out of 50)
  value: '39.7'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 24
  location: Table 2
- id: r_gem_fence
  evaluation_id: ev_gem_fence
  quantity: R1
  coverage: direct
  metric_as_printed: Average Success Count (out of 50)
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 24
  location: Table 2
- id: r_cla_base
  evaluation_id: ev_cla_base
  quantity: R1
  coverage: direct
  metric_as_printed: Average Success Count (out of 50)
  value: '47.0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 24
  location: Table 2
- id: r_cla_fence
  evaluation_id: ev_cla_fence
  quantity: R1
  coverage: direct
  metric_as_printed: Average Success Count (out of 50)
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 24
  location: Table 2
comparisons:
- id: c_gem
  left: ev_gem_fence
  right: ev_gem_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 24
- id: c_cla
  left: ev_cla_fence
  right: ev_cla_base
  purpose: W
  c1_base: matched
  c2_layout: unmatched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 24
claims:
- id: k1
  text: 'simulated fence awareness prevented all injections: 86.7% to 0% over 300 cases'
  location: abstract
  anchor: simulated awareness through prompt instructions achieved complete prevention of injection attacks
  jobs:
  - R1
  supported_by:
  - c_gem
  - c_cla
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2

`ldd_arxiv2025` (2511.21752)

In [ ]:
%%writefile reports/records/ldd_arxiv2025.yaml
report:
  id: ldd_arxiv2025
  slug: '2511.21752'
  title: 'Semantics as a Shield: Label Disguise Defense (LDD) against Prompt Injection in LLM Sentiment Classification'
  first_author: Yanxi Li
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2511.21752'
  version_read: arXiv v3
  first_date: '2025-11-23'
  read: F
  role: proposes
  code_release: not checked
  extraction_scope: 'whole report (8 pages): Table 2 (attacked few-shot mean accuracy, native labels against the sentiment-associated alias group, all nine configurations); the abstract claim (the other-alias group, single
    pairs in Table 1 and the zero-shot results of Table 3 not coded)'
defences:
- id: ldd_m0
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: GPT-5
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m1
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: GPT-4o
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m2
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m3
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: LLaMA 3.2
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m4
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: LLaMA 3.2 (1B)
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m5
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: mistral-large
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m6
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: mistral-small
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m7
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: Gemma3
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
- id: ldd_m8
  name: Label Disguise Defense, mean of four sentiment-associated label pairs
  origin: this_report
  base_checkpoint: Gemma3 (12B)
  axis1:
  - encodings
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - output labels replaced by aliases (heaven/hell, green/red, good/bad, happy/sad) taught by few-shot demonstrations
  - predictions mapped back to positive and negative
  layout: few-shot prompt with alias labels, then the attacked review
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: mean of four sentiment-associated pairs
evaluations:
- id: ev_m0_native
  defence_id: none:GPT-5, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m0_ldd
  defence_id: ldd_m0
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m1_native
  defence_id: none:GPT-4o, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m1_ldd
  defence_id: ldd_m1
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m2_native
  defence_id: none:GPT-4o-mini, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m2_ldd
  defence_id: ldd_m2
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m3_native
  defence_id: none:LLaMA 3.2, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m3_ldd
  defence_id: ldd_m3
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m4_native
  defence_id: none:LLaMA 3.2 (1B), native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m4_ldd
  defence_id: ldd_m4
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m5_native
  defence_id: none:mistral-large, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m5_ldd
  defence_id: ldd_m5
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m6_native
  defence_id: none:mistral-small, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m6_ldd
  defence_id: ldd_m6
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m7_native
  defence_id: none:Gemma3, native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m7_ldd
  defence_id: ldd_m7
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
- id: ev_m8_native
  defence_id: none:Gemma3 (12B), native-label few-shot prompt
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
  baseline_kind: instruction_prompt_only
- id: ev_m8_ldd
  defence_id: ldd_m8
  benchmark: 200 IMDb reviews with a fixed suffix that requests the wrong class using the original labels
  attacker_type: static
  attacker_access: black_box
  judge: sentiment accuracy after mapping back
  layout_same_as_defence: 'yes'
  measured_by: ldd_arxiv2025
  page: 7
  attacker_budget: fixed class-directive suffix
  sample_size: '200'
  decoding: not reported
results:
- id: r_m0_native
  evaluation_id: ev_m0_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '49.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m0_ldd
  evaluation_id: ev_m0_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '57.97'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m1_native
  evaluation_id: ev_m1_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '88.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m1_ldd
  evaluation_id: ev_m1_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '90.03'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m2_native
  evaluation_id: ev_m2_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '89.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m2_ldd
  evaluation_id: ev_m2_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '83.09'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m3_native
  evaluation_id: ev_m3_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '64.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m3_ldd
  evaluation_id: ev_m3_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '72.66'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m4_native
  evaluation_id: ev_m4_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '50.81'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m4_ldd
  evaluation_id: ev_m4_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '50.02'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m5_native
  evaluation_id: ev_m5_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '84.56'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m5_ldd
  evaluation_id: ev_m5_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '78.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m6_native
  evaluation_id: ev_m6_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '86.81'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m6_ldd
  evaluation_id: ev_m6_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '85.44'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m7_native
  evaluation_id: ev_m7_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '79.31'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m7_ldd
  evaluation_id: ev_m7_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '74.31'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m8_native
  evaluation_id: ev_m8_native
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '85.19'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
- id: r_m8_ldd
  evaluation_id: ev_m8_ldd
  quantity: R2attack
  coverage: direct
  metric_as_printed: Attacked few-shot mean accuracy (%)
  value: '81.33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 2
  also_quantity:
  - R1
comparisons:
- id: c_m0
  left: ev_m0_ldd
  right: ev_m0_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m1
  left: ev_m1_ldd
  right: ev_m1_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m2
  left: ev_m2_ldd
  right: ev_m2_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m3
  left: ev_m3_ldd
  right: ev_m3_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m4
  left: ev_m4_ldd
  right: ev_m4_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m5
  left: ev_m5_ldd
  right: ev_m5_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m6
  left: ev_m6_ldd
  right: ev_m6_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m7
  left: ev_m7_ldd
  right: ev_m7_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
- id: c_m8
  left: ev_m8_ldd
  right: ev_m8_native
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the output labels (sentiment-associated aliases against native labels), same demonstrations
  page: 7
claims:
- id: k1
  text: sentiment-associated labels beat other aliases everywhere but beat the native-label baseline in only three of nine configurations
  location: abstract
  anchor: exceed a baseline using the original labels
  jobs:
  - R2
  - R1
  supported_by:
  - c_m0
  - c_m1
  - c_m2
  - c_m3
  - c_m4
  - c_m5
  - c_m6
  - c_m7
  - c_m8
  page: 1
coverage:
  R1: proxy
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R2attack: Table 2 (accuracy on attacked reviews)
    R1: Table 2 (same accuracy read as resistance)

`surfaceheuristics_arxiv2026` (2601.07185)

In [ ]:
%%writefile reports/records/surfaceheuristics_arxiv2026.yaml
report:
  id: surfaceheuristics_arxiv2026
  slug: '2601.07185'
  title: Defenses Against Prompt Attacks Learn Surface Heuristics
  first_author: Shawn Li
  venue: arXiv preprint
  peer_reviewed: 'no'
  doi: 10.18653/v1/2026.acl-long.502
  arxiv: '2601.07185'
  version_read: arXiv v1
  first_date: '2026-01-12'
  read: F
  role: re_measures
  code_release: not checked
  later_version: ACL 2026 (from the search record; not read)
  extraction_scope: 'Sections 3 to 4.2.1: Table 1 (GPQA and MMLU columns) and the second-position columns of Table 2 (GPQA and MMLU), both base models; the abstract claim (AIME, where all accuracies are 0, the prompt guards,
    which act outside the model, Tables 3 to 5 and the appendix not coded)'
defences:
- id: struq_l3
  name: StruQ (fine-tuned by the authors)
  origin: other_report
  base_checkpoint: Llama-3 (instruction-tuned)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: exemplified by SecAlign (Chen et al., 2024) and StrucQ
  name_as_printed: StrucQ
- id: secalign_l3
  name: SecAlign (fine-tuned by the authors)
  origin: other_report
  base_checkpoint: Llama-3 (instruction-tuned)
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: exemplified by SecAlign (Chen et al., 2024) and StrucQ
- id: struq_m
  name: StruQ (fine-tuned by the authors)
  origin: other_report
  base_checkpoint: Mistral (instruction-tuned)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: exemplified by SecAlign (Chen et al., 2024) and StrucQ
  name_as_printed: StrucQ
- id: secalign_m
  name: SecAlign (fine-tuned by the authors)
  origin: other_report
  base_checkpoint: Mistral (instruction-tuned)
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: exemplified by SecAlign (Chen et al., 2024) and StrucQ
evaluations:
- id: ev_t1_l3_base_gpqa
  defence_id: none:Llama-3 (instruction-tuned)
  benchmark: GPQA diamond (100 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t1_l3_base_mmlu
  defence_id: none:Llama-3 (instruction-tuned)
  benchmark: MMLU validation (1,204 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t1_l3_struq_gpqa
  defence_id: struq_l3
  benchmark: GPQA diamond (100 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_l3_struq_mmlu
  defence_id: struq_l3
  benchmark: MMLU validation (1,204 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_l3_secalign_gpqa
  defence_id: secalign_l3
  benchmark: GPQA diamond (100 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_l3_secalign_mmlu
  defence_id: secalign_l3
  benchmark: MMLU validation (1,204 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_m_base_gpqa
  defence_id: none:Mistral (instruction-tuned)
  benchmark: GPQA diamond (100 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t1_m_base_mmlu
  defence_id: none:Mistral (instruction-tuned)
  benchmark: MMLU validation (1,204 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t1_m_struq_gpqa
  defence_id: struq_m
  benchmark: GPQA diamond (100 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_m_struq_mmlu
  defence_id: struq_m
  benchmark: MMLU validation (1,204 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_m_secalign_gpqa
  defence_id: secalign_m
  benchmark: GPQA diamond (100 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t1_m_secalign_mmlu
  defence_id: secalign_m
  benchmark: MMLU validation (1,204 questions), single question
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_l3_base_gpqa
  defence_id: none:Llama-3 (instruction-tuned)
  benchmark: GPQA diamond (100 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t2_l3_base_mmlu
  defence_id: none:Llama-3 (instruction-tuned)
  benchmark: MMLU validation (1,204 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t2_l3_struq_gpqa
  defence_id: struq_l3
  benchmark: GPQA diamond (100 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_l3_struq_mmlu
  defence_id: struq_l3
  benchmark: MMLU validation (1,204 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_l3_secalign_gpqa
  defence_id: secalign_l3
  benchmark: GPQA diamond (100 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_l3_secalign_mmlu
  defence_id: secalign_l3
  benchmark: MMLU validation (1,204 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_m_base_gpqa
  defence_id: none:Mistral (instruction-tuned)
  benchmark: GPQA diamond (100 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t2_m_base_mmlu
  defence_id: none:Mistral (instruction-tuned)
  benchmark: MMLU validation (1,204 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
  baseline_kind: off_the_shelf
- id: ev_t2_m_struq_gpqa
  defence_id: struq_m
  benchmark: GPQA diamond (100 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_m_struq_mmlu
  defence_id: struq_m
  benchmark: MMLU validation (1,204 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_m_secalign_gpqa
  defence_id: secalign_m
  benchmark: GPQA diamond (100 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: '100'
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
- id: ev_t2_m_secalign_mmlu
  defence_id: secalign_m
  benchmark: MMLU validation (1,204 questions), second of two concatenated benign questions
  attacker_type: none
  attacker_access: not_applicable
  judge: accuracy against the answer key; refusal of the benign task
  layout_same_as_defence: 'yes'
  measured_by: surfaceheuristics_arxiv2026
  page: 5
  sample_size: 1,204
  decoding: greedy decoding
  pages:
    benchmark: 5
    decoding: 13
results:
- id: r_t1_l3_base_gpqa_acc
  evaluation_id: ev_t1_l3_base_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '30.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_base_gpqa_rr
  evaluation_id: ev_t1_l3_base_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_base_mmlu_acc
  evaluation_id: ev_t1_l3_base_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '65.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_base_mmlu_rr
  evaluation_id: ev_t1_l3_base_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '2.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_struq_gpqa_acc
  evaluation_id: ev_t1_l3_struq_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '16.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_struq_gpqa_rr
  evaluation_id: ev_t1_l3_struq_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '40.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_struq_mmlu_acc
  evaluation_id: ev_t1_l3_struq_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '32.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_struq_mmlu_rr
  evaluation_id: ev_t1_l3_struq_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '48.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_secalign_gpqa_acc
  evaluation_id: ev_t1_l3_secalign_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '25.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_secalign_gpqa_rr
  evaluation_id: ev_t1_l3_secalign_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '27.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_secalign_mmlu_acc
  evaluation_id: ev_t1_l3_secalign_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '42.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_l3_secalign_mmlu_rr
  evaluation_id: ev_t1_l3_secalign_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '18.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_base_gpqa_acc
  evaluation_id: ev_t1_m_base_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '23.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_base_gpqa_rr
  evaluation_id: ev_t1_m_base_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '20.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_base_mmlu_acc
  evaluation_id: ev_t1_m_base_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '47.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_base_mmlu_rr
  evaluation_id: ev_t1_m_base_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '2.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_struq_gpqa_acc
  evaluation_id: ev_t1_m_struq_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '12.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_struq_gpqa_rr
  evaluation_id: ev_t1_m_struq_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '24.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_struq_mmlu_acc
  evaluation_id: ev_t1_m_struq_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '24.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_struq_mmlu_rr
  evaluation_id: ev_t1_m_struq_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '32.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_secalign_gpqa_acc
  evaluation_id: ev_t1_m_secalign_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '13.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_secalign_gpqa_rr
  evaluation_id: ev_t1_m_secalign_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '40.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_secalign_mmlu_acc
  evaluation_id: ev_t1_m_secalign_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc
  value: '17.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t1_m_secalign_mmlu_rr
  evaluation_id: ev_t1_m_secalign_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (%) refusal of the benign task
  value: '45.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_t2_l3_base_gpqa_acc
  evaluation_id: ev_t2_l3_base_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '32.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_base_gpqa_rr
  evaluation_id: ev_t2_l3_base_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '8.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_base_mmlu_acc
  evaluation_id: ev_t2_l3_base_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '62.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_base_mmlu_rr
  evaluation_id: ev_t2_l3_base_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '4.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_struq_gpqa_acc
  evaluation_id: ev_t2_l3_struq_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '6.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_struq_gpqa_rr
  evaluation_id: ev_t2_l3_struq_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '66.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_struq_mmlu_acc
  evaluation_id: ev_t2_l3_struq_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '5.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_struq_mmlu_rr
  evaluation_id: ev_t2_l3_struq_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '90.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_secalign_gpqa_acc
  evaluation_id: ev_t2_l3_secalign_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '20.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_secalign_gpqa_rr
  evaluation_id: ev_t2_l3_secalign_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '50.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_secalign_mmlu_acc
  evaluation_id: ev_t2_l3_secalign_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '29.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_l3_secalign_mmlu_rr
  evaluation_id: ev_t2_l3_secalign_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '34.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_base_gpqa_acc
  evaluation_id: ev_t2_m_base_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '28.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_base_gpqa_rr
  evaluation_id: ev_t2_m_base_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '16.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_base_mmlu_acc
  evaluation_id: ev_t2_m_base_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '48.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_base_mmlu_rr
  evaluation_id: ev_t2_m_base_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '2.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_struq_gpqa_acc
  evaluation_id: ev_t2_m_struq_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '12.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_struq_gpqa_rr
  evaluation_id: ev_t2_m_struq_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '34.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_struq_mmlu_acc
  evaluation_id: ev_t2_m_struq_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '19.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_struq_mmlu_rr
  evaluation_id: ev_t2_m_struq_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '62.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_secalign_gpqa_acc
  evaluation_id: ev_t2_m_secalign_gpqa
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '8.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_secalign_gpqa_rr
  evaluation_id: ev_t2_m_secalign_gpqa
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '74.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_secalign_mmlu_acc
  evaluation_id: ev_t2_m_secalign_mmlu
  quantity: general_utility
  coverage: direct
  metric_as_printed: Acc (2nd Pos)
  value: '3.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
- id: r_t2_m_secalign_mmlu_rr
  evaluation_id: ev_t2_m_secalign_mmlu
  quantity: other
  coverage: direct
  metric_as_printed: RR (2nd Pos)
  value: '89.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 2
comparisons:
- id: c_t1_l3_struq_gpqa
  left: ev_t1_l3_struq_gpqa
  right: ev_t1_l3_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_l3_struq_mmlu
  left: ev_t1_l3_struq_mmlu
  right: ev_t1_l3_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_l3_secalign_gpqa
  left: ev_t1_l3_secalign_gpqa
  right: ev_t1_l3_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_l3_secalign_mmlu
  left: ev_t1_l3_secalign_mmlu
  right: ev_t1_l3_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_m_struq_gpqa
  left: ev_t1_m_struq_gpqa
  right: ev_t1_m_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_m_struq_mmlu
  left: ev_t1_m_struq_mmlu
  right: ev_t1_m_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_m_secalign_gpqa
  left: ev_t1_m_secalign_gpqa
  right: ev_t1_m_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t1_m_secalign_mmlu
  left: ev_t1_m_secalign_mmlu
  right: ev_t1_m_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_l3_struq_gpqa
  left: ev_t2_l3_struq_gpqa
  right: ev_t2_l3_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_l3_struq_mmlu
  left: ev_t2_l3_struq_mmlu
  right: ev_t2_l3_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_l3_secalign_gpqa
  left: ev_t2_l3_secalign_gpqa
  right: ev_t2_l3_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_l3_secalign_mmlu
  left: ev_t2_l3_secalign_mmlu
  right: ev_t2_l3_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_m_struq_gpqa
  left: ev_t2_m_struq_gpqa
  right: ev_t2_m_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_m_struq_mmlu
  left: ev_t2_m_struq_mmlu
  right: ev_t2_m_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_m_secalign_gpqa
  left: ev_t2_m_secalign_gpqa
  right: ev_t2_m_base_gpqa
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_t2_m_secalign_mmlu
  left: ev_t2_m_secalign_mmlu
  right: ev_t2_m_base_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
claims:
- id: k1
  text: defence fine-tuning makes models reject benign content placed later in a prompt
  location: abstract
  anchor: suffix-task rejection rises from below10%to as high as 90%
  jobs:
  - general_utility
  supported_by:
  - c_t2_l3_struq_gpqa
  - c_t2_l3_struq_mmlu
  - c_t2_l3_secalign_gpqa
  - c_t2_l3_secalign_mmlu
  - c_t2_m_struq_gpqa
  - c_t2_m_struq_mmlu
  - c_t2_m_secalign_gpqa
  - c_t2_m_secalign_mmlu
  page: 1
coverage:
  R1: not_reported
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    general_utility: Tables 1 and 2 (benign benchmarks; refusal rate as other)

`reasalign_arxiv2026` (2601.10173)

In [ ]:
%%writefile reports/records/reasalign_arxiv2026.yaml
report:
  id: reasalign_arxiv2026
  slug: '2601.10173'
  title: 'ReasAlign: Reasoning Enhanced Safety Alignment against Prompt Injection Attack'
  first_author: Hao Li
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2601.10173'
  version_read: arXiv v1
  first_date: '2026-01-15'
  read: F
  role: proposes
  code_release: https://github.com/leolee99/ReasAlign (linked from the abstract; not checked)
  extraction_scope: 'whole report (16 pages): Figure 3 (average column), Figure 4 (all bars), Table 1 (reasoning ablation) and Table 2 (AgentDojo and InjecAgent, both base models); the abstract claims (the four single benchmarks
    of Figure 3, Figure 5 node scaling and Figure 10 token cost not coded)'
defences:
- id: reasalign_l
  name: ReasAlign
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components: &id001
  - LoRA instruction tuning on SQuADv2 samples with injected instructions, whose structured reasoning (problem analysis, reasoning, final answer) was written by GPT-4o-mini from manual guidelines
  - logic judge trained with DPO on reasoning that follows the user (chosen) against reasoning that follows the injection (rejected)
  - 'test-time beam search: N candidate steps sampled, judge picks the best, N = 3 by default'
  layout: user query and external data shown under the labels User Query and External Data in the paper's examples
  layout_compat: &id002
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we adopt Low-Rank Adaptation (LoRA) instruction tuning
  pages: &id003
    components: 4
    layout: 14
    layout_compat: 7
- id: reasalign_q
  name: ReasAlign
  origin: this_report
  base_checkpoint: Qwen2.5-14B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - distillation
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components: *id001
  layout: user query and external data shown under the labels User Query and External Data in the paper's examples
  layout_compat: *id002
  anchor: we adopt Low-Rank Adaptation (LoRA) instruction tuning
  pages: *id003
- id: dirans_l
  name: ReasAlign trained on final answers only (Direct Answer)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - same training data as ReasAlign with final-answer supervision only, no reasoning steps
  anchor: using only final-answer supervi
- id: secalign_l
  name: SecAlign
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  anchor: SecAlign and Meta-SecAlign, which are state-of-the-art defense models
- id: metasecalign_l
  name: Meta SecAlign
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  checkpoint_id: Meta-SecAlign-8B (as printed)
  anchor: SecAlign (Chen et al., 2024b), and Meta-SecAlign-8B
- id: metasecalign_q
  name: Meta SecAlign (reproduced by the authors with the official code)
  origin: other_report
  base_checkpoint: Qwen2.5-14B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: reproduce Meta SecAlign on Qwen2.5-14B-Instruct using its official code
evaluations:
- id: ev_alpaca_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AlpacaEval2, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_alpaca_sec
  defence_id: secalign_l
  benchmark: AlpacaEval2, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_alpaca_meta
  defence_id: metasecalign_l
  benchmark: AlpacaEval2, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_alpaca_ra
  defence_id: reasalign_l
  benchmark: AlpacaEval2, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepclean_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_sepclean_sec
  defence_id: secalign_l
  benchmark: SEP, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepclean_meta
  defence_id: metasecalign_l
  benchmark: SEP, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepclean_ra
  defence_id: reasalign_l
  benchmark: SEP, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_cyse_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: CyberSecEval2 (CySE) prompt-injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly; attack success judging not described
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_cyse_sec
  defence_id: secalign_l
  benchmark: CyberSecEval2 (CySE) prompt-injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly; attack success judging not described
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_cyse_meta
  defence_id: metasecalign_l
  benchmark: CyberSecEval2 (CySE) prompt-injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly; attack success judging not described
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_cyse_ra
  defence_id: reasalign_l
  benchmark: CyberSecEval2 (CySE) prompt-injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly; attack success judging not described
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepatt_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP under prompt injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_sepatt_sec
  defence_id: secalign_l
  benchmark: SEP under prompt injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepatt_meta
  defence_id: metasecalign_l
  benchmark: SEP under prompt injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepatt_ra
  defence_id: reasalign_l
  benchmark: SEP under prompt injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_gk_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: MMLU 0-shot, MMLU-Pro instruct, IFEval and BBH, average
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_gk_sec
  defence_id: secalign_l
  benchmark: MMLU 0-shot, MMLU-Pro instruct, IFEval and BBH, average
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_gk_meta
  defence_id: metasecalign_l
  benchmark: MMLU 0-shot, MMLU-Pro instruct, IFEval and BBH, average
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_gk_ra
  defence_id: reasalign_l
  benchmark: MMLU 0-shot, MMLU-Pro instruct, IFEval and BBH, average
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 7
  decoding: not reported
  pages:
    judge: 6
- id: ev_alpaca_dir
  defence_id: dirans_l
  benchmark: AlpacaEval2, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 8
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepclean_dir
  defence_id: dirans_l
  benchmark: SEP, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 8
  decoding: not reported
  pages:
    judge: 6
- id: ev_cyse_dir
  defence_id: dirans_l
  benchmark: CyberSecEval2 (CySE) prompt-injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 8
  decoding: not reported
  pages:
    judge: 6
- id: ev_sepatt_dir
  defence_id: dirans_l
  benchmark: SEP under prompt injection
  attacker_type: static
  attacker_access: black_box
  judge: GPT-4o-mini judges whether the user's query is answered correctly
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 8
  decoding: not reported
  pages:
    judge: 6
- id: ev_adclean_l_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo, no attack; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_adatt_l_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AgentDojo with injected tool outputs; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_injec_l_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: InjecAgent; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success check
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_adclean_l_meta
  defence_id: metasecalign_l
  benchmark: AgentDojo, no attack; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adatt_l_meta
  defence_id: metasecalign_l
  benchmark: AgentDojo with injected tool outputs; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_injec_l_meta
  defence_id: metasecalign_l
  benchmark: InjecAgent; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success check
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adclean_l_ra
  defence_id: reasalign_l
  benchmark: AgentDojo, no attack; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adatt_l_ra
  defence_id: reasalign_l
  benchmark: AgentDojo with injected tool outputs; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_injec_l_ra
  defence_id: reasalign_l
  benchmark: InjecAgent; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success check
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adclean_q_none
  defence_id: none:Qwen2.5-14B-Instruct
  benchmark: AgentDojo, no attack; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_adatt_q_none
  defence_id: none:Qwen2.5-14B-Instruct
  benchmark: AgentDojo with injected tool outputs; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_injec_q_none
  defence_id: none:Qwen2.5-14B-Instruct
  benchmark: InjecAgent; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success check
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
  baseline_kind: off_the_shelf
- id: ev_adclean_q_meta
  defence_id: metasecalign_q
  benchmark: AgentDojo, no attack; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adatt_q_meta
  defence_id: metasecalign_q
  benchmark: AgentDojo with injected tool outputs; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_injec_q_meta
  defence_id: metasecalign_q
  benchmark: InjecAgent; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success check
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adclean_q_ra
  defence_id: reasalign_q
  benchmark: AgentDojo, no attack; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo task checker
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_adatt_q_ra
  defence_id: reasalign_q
  benchmark: AgentDojo with injected tool outputs; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo task and injection checkers
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
- id: ev_injec_q_ra
  defence_id: reasalign_q
  benchmark: InjecAgent; agents given a commonly used system prompt (Husain, 2024)
  attacker_type: static
  attacker_access: black_box
  judge: InjecAgent attack success check
  layout_same_as_defence: 'yes'
  measured_by: reasalign_arxiv2026
  page: 12
  decoding: not reported
  pages:
    benchmark: 7
results:
- id: r_alpaca_none_general_utility
  evaluation_id: ev_alpaca_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (no attack), AlpacaEval2
  value: '95.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_alpaca_sec_general_utility
  evaluation_id: ev_alpaca_sec
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (no attack), AlpacaEval2
  value: '94.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_alpaca_meta_general_utility
  evaluation_id: ev_alpaca_meta
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (no attack), AlpacaEval2
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_alpaca_ra_general_utility
  evaluation_id: ev_alpaca_ra
  quantity: general_utility
  coverage: direct
  metric_as_printed: Utility (no attack), AlpacaEval2
  value: '96.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (same values as Figure 4)
- id: r_sepclean_none_R2clean
  evaluation_id: ev_sepclean_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility (no attack), SEP
  value: '99.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepclean_sec_R2clean
  evaluation_id: ev_sepclean_sec
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility (no attack), SEP
  value: '98.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepclean_meta_R2clean
  evaluation_id: ev_sepclean_meta
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility (no attack), SEP
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepclean_ra_R2clean
  evaluation_id: ev_sepclean_ra
  quantity: R2clean
  coverage: proxy
  metric_as_printed: Utility (no attack), SEP
  value: '98.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (same values as Figure 4)
- id: r_cyse_none_R2attack
  evaluation_id: ev_cyse_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), CySE
  value: '78.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_cyse_none_R1
  evaluation_id: ev_cyse_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, CySE
  value: '43.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_cyse_sec_R2attack
  evaluation_id: ev_cyse_sec
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), CySE
  value: '92.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_cyse_sec_R1
  evaluation_id: ev_cyse_sec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, CySE
  value: '21.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_cyse_meta_R2attack
  evaluation_id: ev_cyse_meta
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), CySE
  value: '56.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_cyse_meta_R1
  evaluation_id: ev_cyse_meta
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, CySE
  value: '7.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_cyse_ra_R2attack
  evaluation_id: ev_cyse_ra
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), CySE
  value: '94.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (same values as Figure 4)
- id: r_cyse_ra_R1
  evaluation_id: ev_cyse_ra
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, CySE
  value: '3.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (same values as Figure 4)
- id: r_sepatt_none_R2attack
  evaluation_id: ev_sepatt_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), SEP
  value: '77.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepatt_none_R1
  evaluation_id: ev_sepatt_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SEP
  value: '74.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepatt_sec_R2attack
  evaluation_id: ev_sepatt_sec
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), SEP
  value: '96.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepatt_sec_R1
  evaluation_id: ev_sepatt_sec
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SEP
  value: '39.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepatt_meta_R2attack
  evaluation_id: ev_sepatt_meta
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), SEP
  value: '88.8'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepatt_meta_R1
  evaluation_id: ev_sepatt_meta
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SEP
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 4 bar labels
- id: r_sepatt_ra_R2attack
  evaluation_id: ev_sepatt_ra
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Utility (under attack), SEP
  value: '95.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (same values as Figure 4)
- id: r_sepatt_ra_R1
  evaluation_id: ev_sepatt_ra
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SEP
  value: '1.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1 (same values as Figure 4)
- id: r_gk_none_general_utility
  evaluation_id: ev_gk_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: average
  value: '67.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3 bar labels
- id: r_gk_sec_general_utility
  evaluation_id: ev_gk_sec
  quantity: general_utility
  coverage: direct
  metric_as_printed: average
  value: '60.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3 bar labels
- id: r_gk_meta_general_utility
  evaluation_id: ev_gk_meta
  quantity: general_utility
  coverage: direct
  metric_as_printed: average
  value: '66.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3 bar labels
- id: r_gk_ra_general_utility
  evaluation_id: ev_gk_ra
  quantity: general_utility
  coverage: direct
  metric_as_printed: average
  value: '66.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 7
  location: Figure 3 bar labels
- id: r_alpaca_dir_general_utility
  evaluation_id: ev_alpaca_dir
  quantity: general_utility
  coverage: direct
  metric_as_printed: No Attack (Utility), AlpacaEval2
  value: '96.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_sepclean_dir_R2clean
  evaluation_id: ev_sepclean_dir
  quantity: R2clean
  coverage: proxy
  metric_as_printed: No Attack (Utility), SEP
  value: '98.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_cyse_dir_R2attack
  evaluation_id: ev_cyse_dir
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Under Attack (Utility), CySE
  value: '92.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_cyse_dir_R1
  evaluation_id: ev_cyse_dir
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, CySE
  value: '21.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_sepatt_dir_R2attack
  evaluation_id: ev_sepatt_dir
  quantity: R2attack
  coverage: proxy
  metric_as_printed: Under Attack (Utility), SEP
  value: '87.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_sepatt_dir_R1
  evaluation_id: ev_sepatt_dir
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, SEP
  value: '66.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_adclean_l_none
  evaluation_id: ev_adclean_l_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack), Agentdojo
  value: '6.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_l_none_u
  evaluation_id: ev_adatt_l_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack), Agentdojo
  value: '6.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_l_none_asr
  evaluation_id: ev_adatt_l_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Agentdojo
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_injec_l_none
  evaluation_id: ev_injec_l_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, InjecAgent
  value: '13.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adclean_l_meta
  evaluation_id: ev_adclean_l_meta
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack), Agentdojo
  value: '6.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_l_meta_u
  evaluation_id: ev_adatt_l_meta
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack), Agentdojo
  value: '6.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_l_meta_asr
  evaluation_id: ev_adatt_l_meta
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Agentdojo
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_injec_l_meta
  evaluation_id: ev_injec_l_meta
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, InjecAgent
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adclean_l_ra
  evaluation_id: ev_adclean_l_ra
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack), Agentdojo
  value: '10.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_l_ra_u
  evaluation_id: ev_adatt_l_ra
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack), Agentdojo
  value: '7.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_l_ra_asr
  evaluation_id: ev_adatt_l_ra
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Agentdojo
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_injec_l_ra
  evaluation_id: ev_injec_l_ra
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, InjecAgent
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adclean_q_none
  evaluation_id: ev_adclean_q_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack), Agentdojo
  value: '27.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_q_none_u
  evaluation_id: ev_adatt_q_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack), Agentdojo
  value: '20.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_q_none_asr
  evaluation_id: ev_adatt_q_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Agentdojo
  value: '14.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_injec_q_none
  evaluation_id: ev_injec_q_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, InjecAgent
  value: '24.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adclean_q_meta
  evaluation_id: ev_adclean_q_meta
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack), Agentdojo
  value: '25.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_q_meta_u
  evaluation_id: ev_adatt_q_meta
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack), Agentdojo
  value: '20.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_q_meta_asr
  evaluation_id: ev_adatt_q_meta
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Agentdojo
  value: '8.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_injec_q_meta
  evaluation_id: ev_injec_q_meta
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, InjecAgent
  value: '4.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adclean_q_ra
  evaluation_id: ev_adclean_q_ra
  quantity: R2clean
  coverage: direct
  metric_as_printed: Utility (no attack), Agentdojo
  value: '24.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_q_ra_u
  evaluation_id: ev_adatt_q_ra
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility (under attack), Agentdojo
  value: '19.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_adatt_q_ra_asr
  evaluation_id: ev_adatt_q_ra
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Agentdojo
  value: '2.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
- id: r_injec_q_ra
  evaluation_id: ev_injec_q_ra
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, InjecAgent
  value: '2.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 2
comparisons:
- id: c_alpaca_sec
  left: ev_alpaca_sec
  right: ev_alpaca_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_alpaca_ra
  left: ev_alpaca_ra
  right: ev_alpaca_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepclean_sec
  left: ev_sepclean_sec
  right: ev_sepclean_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepclean_ra
  left: ev_sepclean_ra
  right: ev_sepclean_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cyse_sec
  left: ev_cyse_sec
  right: ev_cyse_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cyse_meta
  left: ev_cyse_meta
  right: ev_cyse_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cyse_ra
  left: ev_cyse_ra
  right: ev_cyse_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_cyse_ra_meta
  left: ev_cyse_ra
  right: ev_cyse_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepatt_sec
  left: ev_sepatt_sec
  right: ev_sepatt_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepatt_meta
  left: ev_sepatt_meta
  right: ev_sepatt_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepatt_ra
  left: ev_sepatt_ra
  right: ev_sepatt_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_sepatt_ra_meta
  left: ev_sepatt_ra
  right: ev_sepatt_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gk_sec
  left: ev_gk_sec
  right: ev_gk_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gk_meta
  left: ev_gk_meta
  right: ev_gk_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gk_ra
  left: ev_gk_ra
  right: ev_gk_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_gk_ra_meta
  left: ev_gk_ra
  right: ev_gk_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_alpaca_abl
  left: ev_alpaca_ra
  right: ev_alpaca_dir
  purpose: M
  c1_base: matched
  c2_layout: matched_unstated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: reasoning steps in the training targets (and with them the judge-guided search), same training data
  page: 8
- id: c_sepclean_abl
  left: ev_sepclean_ra
  right: ev_sepclean_dir
  purpose: M
  c1_base: matched
  c2_layout: matched_unstated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: reasoning steps in the training targets (and with them the judge-guided search), same training data
  page: 8
- id: c_cyse_abl
  left: ev_cyse_ra
  right: ev_cyse_dir
  purpose: M
  c1_base: matched
  c2_layout: matched_unstated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: reasoning steps in the training targets (and with them the judge-guided search), same training data
  page: 8
- id: c_sepatt_abl
  left: ev_sepatt_ra
  right: ev_sepatt_dir
  purpose: M
  c1_base: matched
  c2_layout: matched_unstated
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: reasoning steps in the training targets (and with them the judge-guided search), same training data
  page: 8
- id: c_adclean_l_meta
  left: ev_adclean_l_meta
  right: ev_adclean_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adclean_l_ra
  left: ev_adclean_l_ra
  right: ev_adclean_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adclean_l_ra_meta
  left: ev_adclean_l_ra
  right: ev_adclean_l_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adatt_l_meta
  left: ev_adatt_l_meta
  right: ev_adatt_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adatt_l_ra
  left: ev_adatt_l_ra
  right: ev_adatt_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adatt_l_ra_meta
  left: ev_adatt_l_ra
  right: ev_adatt_l_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_injec_l_meta
  left: ev_injec_l_meta
  right: ev_injec_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_injec_l_ra
  left: ev_injec_l_ra
  right: ev_injec_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_injec_l_ra_meta
  left: ev_injec_l_ra
  right: ev_injec_l_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adclean_q_meta
  left: ev_adclean_q_meta
  right: ev_adclean_q_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adclean_q_ra
  left: ev_adclean_q_ra
  right: ev_adclean_q_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adclean_q_ra_meta
  left: ev_adclean_q_ra
  right: ev_adclean_q_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adatt_q_meta
  left: ev_adatt_q_meta
  right: ev_adatt_q_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adatt_q_ra
  left: ev_adatt_q_ra
  right: ev_adatt_q_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_adatt_q_ra_meta
  left: ev_adatt_q_ra
  right: ev_adatt_q_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_injec_q_meta
  left: ev_injec_q_meta
  right: ev_injec_q_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_injec_q_ra
  left: ev_injec_q_ra
  right: ev_injec_q_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_injec_q_ra_meta
  left: ev_injec_q_ra
  right: ev_injec_q_meta
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 12
claims:
- id: k1
  text: on CySE ReasAlign keeps 94.6% utility at 3.6% attack success, far ahead of Meta SecAlign
  location: abstract
  anchor: ReasAlign achieves 94.6% utility and only 3.6% ASR
  jobs:
  - R1
  - R2
  supported_by:
  - c_cyse_ra_meta
  - c_cyse_ra
  - c_cyse_meta
  page: 1
- id: k2
  text: ReasAlign keeps utility comparable to the undefended model
  location: abstract
  anchor: maintains utility comparable to an undefended model
  jobs:
  - general_utility
  - R2
  supported_by:
  - c_gk_ra
  - c_alpaca_ra
  - c_sepclean_ra
  - c_adclean_l_ra
  - c_adclean_q_ra
  page: 1
  contradicted_by:
  - r_adclean_q_ra
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Figure 4 (CySE, SEP), Tables 1 and 2
    R2clean: Table 2 (AgentDojo task checker); SEP utility in Figure 4 is a proxy
    R2attack: Table 2 (AgentDojo); CySE and SEP utility under attack in Figure 4 are proxies
    general_utility: Figure 3, Figure 4 (AlpacaEval2)
schema_gaps:
- where: results with value_source figure_only
  note: values are printed as bar labels inside Figures 3 and 4; value_source has no value for a printed chart label, so they are coded figure_only and the printed value is kept (checked against the page text by the batch
    notebook)
- where: results r_alpaca_meta_general_utility, r_sepclean_meta_R2clean
  note: Figure 4 prints 94.4 (Meta SecAlign, AlpacaEval2) and 96.1 (Meta SecAlign, SEP without attack) as bar labels, but in every text extraction each runs into the next label (94.496.8, 96.198.0), so the values cannot be
    checked against the page text; they are coded figure_only without a value, carry no comparison, and are listed here for a human check (p. 7)
- where: claim k1
  note: the abstract gives Meta SecAlign 56.4% utility and 74.4% ASR on CySE; Figure 4 prints 56.4 for Meta SecAlign's CySE utility but 7.3 for its CySE ASR, and 74.4 is the undefended Llama3 ASR on SEP (p. 7); the coded results
    follow the figure
- where: defence reasalign_l, reasalign_q
  note: preference optimisation trains the separate logic judge used in the test-time search, not the answering model; distillation because the reasoning targets were written by GPT-4o-mini

`redvisor_arxiv2026` (2602.01795)

In [ ]:
%%writefile reports/records/redvisor_arxiv2026.yaml
report:
  id: redvisor_arxiv2026
  slug: '2602.01795'
  title: 'RedVisor: Reasoning-Aware Prompt Injection Defense via Zero-Copy KV Cache Reuse'
  first_author: Mingrui Liu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2602.01795'
  version_read: arXiv v1
  first_date: '2026-02-02'
  read: F
  role: proposes
  code_release: anonymous repository linked in a footnote (p. 2); not checked
  extraction_scope: 'whole report (23 pages): Table 3 (Alpaca-Farm ASR) for the undefended model, Sandwich, StruQ, SecAlign and RedVisor on all three backbones; the three win rates printed in the text of Section 7.6 with the
    other Figure 3 bars as figure_only; the GCG-classifier rows of Table 4; the abstract and introduction claims. Left out: PromptArmor, PromptLocate and DataFilter, which detect or rewrite before an unchanged model; detection
    scores (Tables 2, 5, 6); Table 7 (RedVisor guarding a GPT-4o agent); Table 8 (NQ-simplified); Table 9 (adapter ablation); latency'
defences:
- id: rv_l
  name: RedVisor
  origin: this_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - user query and context in <user query> and <reference context> tags, context split into indexed sentences (p. 3)
  - gated self-attention adapter (about 70M parameters) on the top layer of the frozen backbone, trained to write a reasoning trace that locates injected sentences (templates filled by a spaCy intent extractor)
  - 'phase 2: adapter muted, transition instruction appended, frozen backbone answers with the reasoning trace in context'
  layout: inspection prefix as the first user message, then the tagged user query and context
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we position our adapter exclusively at the top layer
  pages:
    components: 3
    layout: 4
    layout_compat: 22
- id: sw_l
  name: Sandwich Defense
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 19
  components:
  - the user instruction repeated at the end of the user context
  anchor: repeats the user instruction at the end
- id: sq_l
  name: StruQ (trained by the authors with the official scripts)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: We utilize the official training scripts to fine-tune the backbone models
- id: sa_l
  name: SecAlign (reconstructed by the authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: we reconstruct the preference dataset following the data generation pipeline
- id: rv_m
  name: RedVisor
  origin: this_report
  base_checkpoint: Mistral-7B-v0.3-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - user query and context in <user query> and <reference context> tags, context split into indexed sentences (p. 3)
  - gated self-attention adapter (about 70M parameters) on the top layer of the frozen backbone, trained to write a reasoning trace that locates injected sentences (templates filled by a spaCy intent extractor)
  - 'phase 2: adapter muted, transition instruction appended, frozen backbone answers with the reasoning trace in context'
  layout: inspection prefix as the first user message, then the tagged user query and context
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we position our adapter exclusively at the top layer
  pages:
    components: 3
    layout: 4
    layout_compat: 22
- id: sw_m
  name: Sandwich Defense
  origin: other_report
  base_checkpoint: Mistral-7B-v0.3-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 19
  components:
  - the user instruction repeated at the end of the user context
  anchor: repeats the user instruction at the end
- id: sq_m
  name: StruQ (trained by the authors with the official scripts)
  origin: other_report
  base_checkpoint: Mistral-7B-v0.3-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: We utilize the official training scripts to fine-tune the backbone models
- id: sa_m
  name: SecAlign (reconstructed by the authors)
  origin: other_report
  base_checkpoint: Mistral-7B-v0.3-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: we reconstruct the preference dataset following the data generation pipeline
- id: rv_q
  name: RedVisor
  origin: this_report
  base_checkpoint: Qwen-2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - user query and context in <user query> and <reference context> tags, context split into indexed sentences (p. 3)
  - gated self-attention adapter (about 70M parameters) on the top layer of the frozen backbone, trained to write a reasoning trace that locates injected sentences (templates filled by a spaCy intent extractor)
  - 'phase 2: adapter muted, transition instruction appended, frozen backbone answers with the reasoning trace in context'
  layout: inspection prefix as the first user message, then the tagged user query and context
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we position our adapter exclusively at the top layer
  pages:
    components: 3
    layout: 4
    layout_compat: 22
- id: sw_q
  name: Sandwich Defense
  origin: other_report
  base_checkpoint: Qwen-2.5-7B-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 19
  components:
  - the user instruction repeated at the end of the user context
  anchor: repeats the user instruction at the end
- id: sq_q
  name: StruQ (trained by the authors with the official scripts)
  origin: other_report
  base_checkpoint: Qwen-2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: We utilize the official training scripts to fine-tune the backbone models
- id: sa_q
  name: SecAlign (reconstructed by the authors)
  origin: other_report
  base_checkpoint: Qwen-2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 19
  anchor: we reconstruct the preference dataset following the data generation pipeline
evaluations:
- id: ev_l_none_naive
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_l_none_ignore
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_l_none_esc
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_l_none_comp
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_l_none_multi
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_l_none_gcg
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_l_sw_naive
  defence_id: sw_l
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sw_ignore
  defence_id: sw_l
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sw_esc
  defence_id: sw_l
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sw_comp
  defence_id: sw_l
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sw_multi
  defence_id: sw_l
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sw_gcg
  defence_id: sw_l
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sq_naive
  defence_id: sq_l
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sq_ignore
  defence_id: sq_l
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sq_esc
  defence_id: sq_l
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sq_comp
  defence_id: sq_l
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sq_multi
  defence_id: sq_l
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sq_gcg
  defence_id: sq_l
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sa_naive
  defence_id: sa_l
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sa_ignore
  defence_id: sa_l
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sa_esc
  defence_id: sa_l
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sa_comp
  defence_id: sa_l
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sa_multi
  defence_id: sa_l
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_sa_gcg
  defence_id: sa_l
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_rv_naive
  defence_id: rv_l
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_rv_ignore
  defence_id: rv_l
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_rv_esc
  defence_id: rv_l
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_rv_comp
  defence_id: rv_l
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_rv_multi
  defence_id: rv_l
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_l_rv_gcg
  defence_id: rv_l
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_none_naive
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_m_none_ignore
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_m_none_esc
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_m_none_comp
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_m_none_multi
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_m_none_gcg
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_m_sw_naive
  defence_id: sw_m
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sw_ignore
  defence_id: sw_m
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sw_esc
  defence_id: sw_m
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sw_comp
  defence_id: sw_m
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sw_multi
  defence_id: sw_m
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sw_gcg
  defence_id: sw_m
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sq_naive
  defence_id: sq_m
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sq_ignore
  defence_id: sq_m
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sq_esc
  defence_id: sq_m
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sq_comp
  defence_id: sq_m
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sq_multi
  defence_id: sq_m
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sq_gcg
  defence_id: sq_m
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sa_naive
  defence_id: sa_m
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sa_ignore
  defence_id: sa_m
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sa_esc
  defence_id: sa_m
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sa_comp
  defence_id: sa_m
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sa_multi
  defence_id: sa_m
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_sa_gcg
  defence_id: sa_m
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_rv_naive
  defence_id: rv_m
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_rv_ignore
  defence_id: rv_m
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_rv_esc
  defence_id: rv_m
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_rv_comp
  defence_id: rv_m
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_rv_multi
  defence_id: rv_m
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_m_rv_gcg
  defence_id: rv_m
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_none_naive
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_q_none_ignore
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_q_none_esc
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_q_none_comp
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_q_none_multi
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_q_none_gcg
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
  baseline_kind: off_the_shelf
- id: ev_q_sw_naive
  defence_id: sw_q
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sw_ignore
  defence_id: sw_q
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sw_esc
  defence_id: sw_q
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sw_comp
  defence_id: sw_q
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sw_multi
  defence_id: sw_q
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sw_gcg
  defence_id: sw_q
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sq_naive
  defence_id: sq_q
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sq_ignore
  defence_id: sq_q
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sq_esc
  defence_id: sq_q
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sq_comp
  defence_id: sq_q
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sq_multi
  defence_id: sq_q
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sq_gcg
  defence_id: sq_q
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sa_naive
  defence_id: sa_q
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sa_ignore
  defence_id: sa_q
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sa_esc
  defence_id: sa_q
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sa_comp
  defence_id: sa_q
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sa_multi
  defence_id: sa_q
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_sa_gcg
  defence_id: sa_q
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_rv_naive
  defence_id: rv_q
  benchmark: Alpaca-Farm with Naive injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_rv_ignore
  defence_id: rv_q
  benchmark: Alpaca-Farm with Ignore injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_rv_esc
  defence_id: rv_q
  benchmark: Alpaca-Farm with Escape-character injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_rv_comp
  defence_id: rv_q
  benchmark: Alpaca-Farm with Completion injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_rv_multi
  defence_id: rv_q
  benchmark: Alpaca-Farm with Multi-round injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: static
  attacker_access: black_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_q_rv_gcg
  defence_id: rv_q
  benchmark: Alpaca-Farm with GCG suffix optimised on the model (GCG-LLM) injection at a random position, payload 'Print exactly <UUID>'
  attacker_type: optimisation
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 7
  decoding: not reported
  pages:
    benchmark: 19
    judge: 18
- id: ev_util_l_none
  defence_id: none:Llama-3-8B-Instruct
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_util_l_sq
  defence_id: sq_l
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_l_sa
  defence_id: sa_l
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_l_rv
  defence_id: rv_l
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_m_none
  defence_id: none:Mistral-7B-v0.3-Instruct
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_util_m_sq
  defence_id: sq_m
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_m_sa
  defence_id: sa_m
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_m_rv
  defence_id: rv_m
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_q_none
  defence_id: none:Qwen-2.5-7B-Instruct
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_util_q_sq
  defence_id: sq_q
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_q_sa
  defence_id: sa_q
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_util_q_rv
  defence_id: rv_q
  benchmark: AlpacaEval 2.0, clean samples
  attacker_type: none
  attacker_access: not_applicable
  judge: Qwen3-Max as judge (win rate)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 8
  decoding: not reported
- id: ev_l_rv_gcgcls
  defence_id: rv_l
  benchmark: Alpaca-Farm with a GCG suffix optimised to make the adapter output 'No injection detected' (GCG-classifier)
  attacker_type: adaptive
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 21
  decoding: not reported
  pages:
    judge: 18
- id: ev_m_rv_gcgcls
  defence_id: rv_m
  benchmark: Alpaca-Farm with a GCG suffix optimised to make the adapter output 'No injection detected' (GCG-classifier)
  attacker_type: adaptive
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 21
  decoding: not reported
  pages:
    judge: 18
- id: ev_q_rv_gcgcls
  defence_id: rv_q
  benchmark: Alpaca-Farm with a GCG suffix optimised to make the adapter output 'No injection detected' (GCG-classifier)
  attacker_type: adaptive
  attacker_access: white_box
  judge: target payload printed (string match on a random UUID)
  layout_same_as_defence: 'yes'
  measured_by: redvisor_arxiv2026
  page: 21
  decoding: not reported
  pages:
    judge: 18
results:
- id: r_l_none_naive
  evaluation_id: ev_l_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.58'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_none_ignore
  evaluation_id: ev_l_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.71'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_none_esc
  evaluation_id: ev_l_none_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.62'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_none_comp
  evaluation_id: ev_l_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.71'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_none_multi
  evaluation_id: ev_l_none_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_none_gcg
  evaluation_id: ev_l_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.68'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sw_naive
  evaluation_id: ev_l_sw_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.38'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sw_ignore
  evaluation_id: ev_l_sw_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.51'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sw_esc
  evaluation_id: ev_l_sw_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.36'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sw_comp
  evaluation_id: ev_l_sw_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.45'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sw_multi
  evaluation_id: ev_l_sw_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.42'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sw_gcg
  evaluation_id: ev_l_sw_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.40'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sq_naive
  evaluation_id: ev_l_sq_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sq_ignore
  evaluation_id: ev_l_sq_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sq_esc
  evaluation_id: ev_l_sq_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sq_comp
  evaluation_id: ev_l_sq_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.10'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sq_multi
  evaluation_id: ev_l_sq_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.14'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sq_gcg
  evaluation_id: ev_l_sq_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.51'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sa_naive
  evaluation_id: ev_l_sa_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sa_ignore
  evaluation_id: ev_l_sa_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sa_esc
  evaluation_id: ev_l_sa_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sa_comp
  evaluation_id: ev_l_sa_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.09'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sa_multi
  evaluation_id: ev_l_sa_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.11'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_sa_gcg
  evaluation_id: ev_l_sa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.44'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_rv_naive
  evaluation_id: ev_l_rv_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_rv_ignore
  evaluation_id: ev_l_rv_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_rv_esc
  evaluation_id: ev_l_rv_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_rv_comp
  evaluation_id: ev_l_rv_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_rv_multi
  evaluation_id: ev_l_rv_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l_rv_gcg
  evaluation_id: ev_l_rv_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_none_naive
  evaluation_id: ev_m_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.64'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_none_ignore
  evaluation_id: ev_m_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.66'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_none_esc
  evaluation_id: ev_m_none_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_none_comp
  evaluation_id: ev_m_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_none_multi
  evaluation_id: ev_m_none_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.77'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_none_gcg
  evaluation_id: ev_m_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.78'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sw_naive
  evaluation_id: ev_m_sw_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.42'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sw_ignore
  evaluation_id: ev_m_sw_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sw_esc
  evaluation_id: ev_m_sw_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.43'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sw_comp
  evaluation_id: ev_m_sw_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.55'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sw_multi
  evaluation_id: ev_m_sw_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sw_gcg
  evaluation_id: ev_m_sw_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.48'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sq_naive
  evaluation_id: ev_m_sq_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sq_ignore
  evaluation_id: ev_m_sq_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.08'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sq_esc
  evaluation_id: ev_m_sq_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.07'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sq_comp
  evaluation_id: ev_m_sq_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.12'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sq_multi
  evaluation_id: ev_m_sq_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.15'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sq_gcg
  evaluation_id: ev_m_sq_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.58'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sa_naive
  evaluation_id: ev_m_sa_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sa_ignore
  evaluation_id: ev_m_sa_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sa_esc
  evaluation_id: ev_m_sa_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sa_comp
  evaluation_id: ev_m_sa_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.10'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sa_multi
  evaluation_id: ev_m_sa_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.15'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_sa_gcg
  evaluation_id: ev_m_sa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.50'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_rv_naive
  evaluation_id: ev_m_rv_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_rv_ignore
  evaluation_id: ev_m_rv_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_rv_esc
  evaluation_id: ev_m_rv_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_rv_comp
  evaluation_id: ev_m_rv_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_rv_multi
  evaluation_id: ev_m_rv_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.07'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_m_rv_gcg
  evaluation_id: ev_m_rv_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_none_naive
  evaluation_id: ev_q_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.60'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_none_ignore
  evaluation_id: ev_q_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_none_esc
  evaluation_id: ev_q_none_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.58'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_none_comp
  evaluation_id: ev_q_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.71'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_none_multi
  evaluation_id: ev_q_none_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.71'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_none_gcg
  evaluation_id: ev_q_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sw_naive
  evaluation_id: ev_q_sw_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.36'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sw_ignore
  evaluation_id: ev_q_sw_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.54'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sw_esc
  evaluation_id: ev_q_sw_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.38'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sw_comp
  evaluation_id: ev_q_sw_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.51'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sw_multi
  evaluation_id: ev_q_sw_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.58'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sw_gcg
  evaluation_id: ev_q_sw_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.46'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sq_naive
  evaluation_id: ev_q_sq_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sq_ignore
  evaluation_id: ev_q_sq_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.06'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sq_esc
  evaluation_id: ev_q_sq_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sq_comp
  evaluation_id: ev_q_sq_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.11'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sq_multi
  evaluation_id: ev_q_sq_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.16'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sq_gcg
  evaluation_id: ev_q_sq_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.47'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sa_naive
  evaluation_id: ev_q_sa_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sa_ignore
  evaluation_id: ev_q_sa_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sa_esc
  evaluation_id: ev_q_sa_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sa_comp
  evaluation_id: ev_q_sa_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.08'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sa_multi
  evaluation_id: ev_q_sa_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.13'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_sa_gcg
  evaluation_id: ev_q_sa_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.42'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_rv_naive
  evaluation_id: ev_q_rv_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Naive)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_rv_ignore
  evaluation_id: ev_q_rv_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Ignore)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_rv_esc
  evaluation_id: ev_q_rv_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Escape)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_rv_comp
  evaluation_id: ev_q_rv_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Completion)
  value: '0.02'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_rv_multi
  evaluation_id: ev_q_rv_multi
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (Multi)
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q_rv_gcg
  evaluation_id: ev_q_rv_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGLLM)
  value: '0.00'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_util_l_none
  evaluation_id: ev_util_l_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: win rate
  value: '87.27'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 8
  location: Section 7.6 text
- id: r_util_l_sq
  evaluation_id: ev_util_l_sq
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_l_sa
  evaluation_id: ev_util_l_sa
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_l_rv
  evaluation_id: ev_util_l_rv
  quantity: general_utility
  coverage: direct
  metric_as_printed: win rate
  value: '85.78'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 8
  location: Section 7.6 text
- id: r_util_m_none
  evaluation_id: ev_util_m_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_m_sq
  evaluation_id: ev_util_m_sq
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_m_sa
  evaluation_id: ev_util_m_sa
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_m_rv
  evaluation_id: ev_util_m_rv
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_q_none
  evaluation_id: ev_util_q_none
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_q_sq
  evaluation_id: ev_util_q_sq
  quantity: general_utility
  coverage: direct
  metric_as_printed: win rate
  value: '56.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 8
  location: Section 7.6 text
- id: r_util_q_sa
  evaluation_id: ev_util_q_sa
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_util_q_rv
  evaluation_id: ev_util_q_rv
  quantity: general_utility
  coverage: direct
  metric_as_printed: WinRate (%)
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: figure_only
  page: 8
  location: Figure 3
- id: r_l_rv_gcgcls
  evaluation_id: ev_l_rv_gcgcls
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGclassifier)
  value: '0.05'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_m_rv_gcgcls
  evaluation_id: ev_m_rv_gcgcls
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGclassifier)
  value: '0.07'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
- id: r_q_rv_gcgcls
  evaluation_id: ev_q_rv_gcgcls
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (GCGclassifier)
  value: '0.04'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 4
comparisons:
- id: c_l_sw_naive
  left: ev_l_sw_naive
  right: ev_l_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sw_ignore
  left: ev_l_sw_ignore
  right: ev_l_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sw_esc
  left: ev_l_sw_esc
  right: ev_l_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sw_comp
  left: ev_l_sw_comp
  right: ev_l_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sw_multi
  left: ev_l_sw_multi
  right: ev_l_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sw_gcg
  left: ev_l_sw_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sq_naive
  left: ev_l_sq_naive
  right: ev_l_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sq_ignore
  left: ev_l_sq_ignore
  right: ev_l_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sq_esc
  left: ev_l_sq_esc
  right: ev_l_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sq_comp
  left: ev_l_sq_comp
  right: ev_l_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sq_multi
  left: ev_l_sq_multi
  right: ev_l_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sq_gcg
  left: ev_l_sq_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sa_naive
  left: ev_l_sa_naive
  right: ev_l_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sa_ignore
  left: ev_l_sa_ignore
  right: ev_l_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sa_esc
  left: ev_l_sa_esc
  right: ev_l_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sa_comp
  left: ev_l_sa_comp
  right: ev_l_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sa_multi
  left: ev_l_sa_multi
  right: ev_l_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_sa_gcg
  left: ev_l_sa_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_naive
  left: ev_l_rv_naive
  right: ev_l_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_ignore
  left: ev_l_rv_ignore
  right: ev_l_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_esc
  left: ev_l_rv_esc
  right: ev_l_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_comp
  left: ev_l_rv_comp
  right: ev_l_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_multi
  left: ev_l_rv_multi
  right: ev_l_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_gcg
  left: ev_l_rv_gcg
  right: ev_l_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_sa_naive
  left: ev_l_rv_naive
  right: ev_l_sa_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_sa_ignore
  left: ev_l_rv_ignore
  right: ev_l_sa_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_sa_esc
  left: ev_l_rv_esc
  right: ev_l_sa_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_sa_comp
  left: ev_l_rv_comp
  right: ev_l_sa_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_sa_multi
  left: ev_l_rv_multi
  right: ev_l_sa_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l_rv_sa_gcg
  left: ev_l_rv_gcg
  right: ev_l_sa_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sw_naive
  left: ev_m_sw_naive
  right: ev_m_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sw_ignore
  left: ev_m_sw_ignore
  right: ev_m_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sw_esc
  left: ev_m_sw_esc
  right: ev_m_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sw_comp
  left: ev_m_sw_comp
  right: ev_m_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sw_multi
  left: ev_m_sw_multi
  right: ev_m_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sw_gcg
  left: ev_m_sw_gcg
  right: ev_m_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sq_naive
  left: ev_m_sq_naive
  right: ev_m_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sq_ignore
  left: ev_m_sq_ignore
  right: ev_m_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sq_esc
  left: ev_m_sq_esc
  right: ev_m_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sq_comp
  left: ev_m_sq_comp
  right: ev_m_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sq_multi
  left: ev_m_sq_multi
  right: ev_m_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sq_gcg
  left: ev_m_sq_gcg
  right: ev_m_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sa_naive
  left: ev_m_sa_naive
  right: ev_m_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sa_ignore
  left: ev_m_sa_ignore
  right: ev_m_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sa_esc
  left: ev_m_sa_esc
  right: ev_m_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sa_comp
  left: ev_m_sa_comp
  right: ev_m_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sa_multi
  left: ev_m_sa_multi
  right: ev_m_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_sa_gcg
  left: ev_m_sa_gcg
  right: ev_m_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_naive
  left: ev_m_rv_naive
  right: ev_m_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_ignore
  left: ev_m_rv_ignore
  right: ev_m_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_esc
  left: ev_m_rv_esc
  right: ev_m_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_comp
  left: ev_m_rv_comp
  right: ev_m_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_multi
  left: ev_m_rv_multi
  right: ev_m_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_gcg
  left: ev_m_rv_gcg
  right: ev_m_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_sa_naive
  left: ev_m_rv_naive
  right: ev_m_sa_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_sa_ignore
  left: ev_m_rv_ignore
  right: ev_m_sa_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_sa_esc
  left: ev_m_rv_esc
  right: ev_m_sa_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_sa_comp
  left: ev_m_rv_comp
  right: ev_m_sa_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_sa_multi
  left: ev_m_rv_multi
  right: ev_m_sa_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_m_rv_sa_gcg
  left: ev_m_rv_gcg
  right: ev_m_sa_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sw_naive
  left: ev_q_sw_naive
  right: ev_q_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sw_ignore
  left: ev_q_sw_ignore
  right: ev_q_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sw_esc
  left: ev_q_sw_esc
  right: ev_q_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sw_comp
  left: ev_q_sw_comp
  right: ev_q_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sw_multi
  left: ev_q_sw_multi
  right: ev_q_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sw_gcg
  left: ev_q_sw_gcg
  right: ev_q_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sq_naive
  left: ev_q_sq_naive
  right: ev_q_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sq_ignore
  left: ev_q_sq_ignore
  right: ev_q_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sq_esc
  left: ev_q_sq_esc
  right: ev_q_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sq_comp
  left: ev_q_sq_comp
  right: ev_q_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sq_multi
  left: ev_q_sq_multi
  right: ev_q_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sq_gcg
  left: ev_q_sq_gcg
  right: ev_q_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sa_naive
  left: ev_q_sa_naive
  right: ev_q_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sa_ignore
  left: ev_q_sa_ignore
  right: ev_q_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sa_esc
  left: ev_q_sa_esc
  right: ev_q_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sa_comp
  left: ev_q_sa_comp
  right: ev_q_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sa_multi
  left: ev_q_sa_multi
  right: ev_q_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_sa_gcg
  left: ev_q_sa_gcg
  right: ev_q_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_naive
  left: ev_q_rv_naive
  right: ev_q_none_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_ignore
  left: ev_q_rv_ignore
  right: ev_q_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_esc
  left: ev_q_rv_esc
  right: ev_q_none_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_comp
  left: ev_q_rv_comp
  right: ev_q_none_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_multi
  left: ev_q_rv_multi
  right: ev_q_none_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_gcg
  left: ev_q_rv_gcg
  right: ev_q_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_sa_naive
  left: ev_q_rv_naive
  right: ev_q_sa_naive
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_sa_ignore
  left: ev_q_rv_ignore
  right: ev_q_sa_ignore
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_sa_esc
  left: ev_q_rv_esc
  right: ev_q_sa_esc
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_sa_comp
  left: ev_q_rv_comp
  right: ev_q_sa_comp
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_sa_multi
  left: ev_q_rv_multi
  right: ev_q_sa_multi
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q_rv_sa_gcg
  left: ev_q_rv_gcg
  right: ev_q_sa_gcg
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_util_l_rv
  left: ev_util_l_rv
  right: ev_util_l_none
  purpose: W
  c1_base: matched
  c2_layout: unreported
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: RedVisor beats existing defences in detection accuracy and throughput with negligible utility loss
  location: abstract
  anchor: outperforms state-of-the-art defenses in detection accuracy and throughput
  jobs:
  - general_utility
  supported_by:
  - c_util_l_rv
  page: 1
- id: k2
  text: RedVisor gives much lower attack success than existing methods while keeping utility
  location: introduction
  anchor: significantly lower attack success rates
  jobs:
  - R1
  - general_utility
  supported_by:
  - c_l_rv_naive
  - c_l_rv_ignore
  - c_l_rv_esc
  - c_l_rv_comp
  - c_l_rv_multi
  - c_l_rv_gcg
  - c_l_rv_sa_naive
  - c_l_rv_sa_ignore
  - c_l_rv_sa_esc
  - c_l_rv_sa_comp
  - c_l_rv_sa_multi
  - c_l_rv_sa_gcg
  - c_m_rv_naive
  - c_m_rv_ignore
  - c_m_rv_esc
  - c_m_rv_comp
  - c_m_rv_multi
  - c_m_rv_gcg
  - c_m_rv_sa_naive
  - c_m_rv_sa_ignore
  - c_m_rv_sa_esc
  - c_m_rv_sa_comp
  - c_m_rv_sa_multi
  - c_m_rv_sa_gcg
  - c_q_rv_naive
  - c_q_rv_ignore
  - c_q_rv_esc
  - c_q_rv_comp
  - c_q_rv_multi
  - c_q_rv_gcg
  - c_q_rv_sa_naive
  - c_q_rv_sa_ignore
  - c_q_rv_sa_esc
  - c_q_rv_sa_comp
  - c_q_rv_sa_multi
  - c_q_rv_sa_gcg
  - c_util_l_rv
  page: 2
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3 (Alpaca-Farm), Table 4
    general_utility: Figure 3 and Section 7.6
schema_gaps:
- where: evaluations ev_*_none_*, ev_*_rv_*
  note: Section 6.1 says the five attack categories (Naive, Completion, Multi-round, Ignore, Escape) were used to build RedVisor's training data and are the same five used in Table 3; Section 7.3 says the tested patterns were
    unseen in training, and Appendix E.1 says the payload is a random UUID unseen in training (pp. 6, 7, 19)
- where: results r_util_*
  note: Appendix D.2 says WinRate compares RedVisor's responses against the vanilla backbone, but Section 7.6 prints a win rate of 87.27% for the undefended Llama-3 backbone itself; the reference model of the win rate is not
    clear (pp. 8, 19)

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04c"
cfg_exclusions = 0
cfg_next = "batch 04d"
IDS = ['civ_arxiv2025', 'advft2022_arxiv2025', 'secinfer_arxiv2025', 'rlhammer_arxiv2025', 'arlas_arxiv2025', 'attackermovessecond_arxiv2025', 'verih_arxiv2025', 'promptfencing_arxiv2025', 'ldd_arxiv2025', 'surfaceheuristics_arxiv2026', 'reasalign_arxiv2026', 'redvisor_arxiv2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in man.values() if m.get("decision") == "include")
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04c extraction batch c: twelve included studies coded from their PDFs, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)